In [1]:
"""
UC4 Structured Micro-Priority Rule Registry Validation Notebook

Purpose
-------
This notebook validates a production-aligned UC4 Adaptive Care Focus Checklist /
Personalized Care Priorities Engine.

It replaces the prototype's template-specific imperative rule blocks with:

1. Strict normalized rule contexts
2. Declarative rule registry
3. Generic rule validator engine
4. Typed fired-rule records
5. Normalized scoring trace
6. Template-registry-based rendering
7. Structured caregiver response schemas
8. UC1/UC2 safety compatibility checks
9. JSON/CSV artifacts for TypeScript handoff

Important Safety Boundaries
---------------------------
UC4 does NOT:
- diagnose
- detect seizures
- detect wounds
- measure tone/spasticity
- infer medication side effects as fact
- infer medication causality
- recommend treatment changes
- recommend medication changes
- suppress UC1/UC2 emergency workflows
- use free text for scoring
- use SLM output for scoring

UC4 DOES:
- generate structured caregiver micro-priorities
- use rule/template logic, not open-ended generation
- ask for structured observations
- create provider-ready summaries from structured evidence
- support caregiver follow-up and next-cycle personalization
"""

"\nUC4 Structured Micro-Priority Rule Registry Validation Notebook\n\nPurpose\n-------\nThis notebook validates a production-aligned UC4 Adaptive Care Focus Checklist /\nPersonalized Care Priorities Engine.\n\nIt replaces the prototype's template-specific imperative rule blocks with:\n\n1. Strict normalized rule contexts\n2. Declarative rule registry\n3. Generic rule validator engine\n4. Typed fired-rule records\n5. Normalized scoring trace\n6. Template-registry-based rendering\n7. Structured caregiver response schemas\n8. UC1/UC2 safety compatibility checks\n9. JSON/CSV artifacts for TypeScript handoff\n\nImportant Safety Boundaries\n---------------------------\nUC4 does NOT:\n- diagnose\n- detect seizures\n- detect wounds\n- measure tone/spasticity\n- infer medication side effects as fact\n- infer medication causality\n- recommend treatment changes\n- recommend medication changes\n- suppress UC1/UC2 emergency workflows\n- use free text for scoring\n- use SLM output for scoring\n\nUC4

In [3]:
from __future__ import annotations

import json
import math
import os
import re
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Callable, Dict, List, Optional, Tuple

import pandas as pd

In [5]:
OUTPUT_DIR = Path("uc4_rule_registry_validation_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now(timezone.utc).isoformat()

SCHEMA_VERSION = "uc4_schema_v0.1.0"
TEMPLATE_REGISTRY_VERSION = "uc4_template_registry_v0.1.0"
RULE_REGISTRY_VERSION = "uc4_rule_registry_v0.1.0"
SCORING_VERSION = "uc4_scoring_v0.1.0"
ENGINE_VERSION = "uc4_structured_micropriority_engine_v0.1.0"

def write_json(path: Path, obj: Any):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def clamp01(x: float) -> float:
    return max(0.0, min(1.0, float(x)))

def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

In [7]:
OBSERVATION_CODES = [
    "LOOKS_NORMAL",
    "NOT_SURE",
    "DEVICE_OR_SENSOR_ISSUE",
    "RECENT_ACTIVITY_OR_EXERTION",
    "LOW_MOVEMENT",
    "TRANSFER_OR_POSITIONING_CONTEXT",
    "PAIN_OR_DISCOMFORT",
    "UNUSUAL_FATIGUE",
    "POOR_SLEEP_OR_RESTLESSNESS",
    "BREATHING_CONCERN",
    "COLOR_OR_OXYGEN_CONCERN",
    "MISSED_OR_DELAYED_MEDICATION",
    "RECENT_MEDICATION_CHANGE",
    "APPETITE_OR_HYDRATION_CHANGE",
    "BOWEL_OR_BLADDER_CHANGE",
    "SKIN_OR_PRESSURE_CONCERN",
    "SEIZURE_LIKE_EVENT_REPORTED",
    "UNUSUAL_RESPONSIVENESS",
    "CAREGIVER_WANTS_PROVIDER_REVIEW",
    "FALL_OR_NEAR_FALL",
    "THERAPY_ROUTINE_DIFFICULTY",
]

CONTEXT_CODES = [
    "DURING_TRANSFER",
    "WHILE_SITTING_OR_POSITIONED",
    "AFTER_ACTIVITY_OR_THERAPY",
    "AROUND_MEDICATION_TIME",
    "DURING_SLEEP_OR_NIGHT",
    "MEAL_OR_HYDRATION_RELATED",
    "BATHROOM_OR_BOWEL_BLADDER",
    "UNKNOWN_OR_NOT_SURE",
]

MEDICATION_WATCH_AREA_CODES = [
    "SLEEPINESS_FATIGUE",
    "DIZZINESS_OR_LIGHTHEADEDNESS",
    "WEAKNESS_OR_LOW_TONE_CONCERN",
    "MOOD_BEHAVIOR_CHANGE",
    "APPETITE_OR_HYDRATION_CHANGE",
    "BOWEL_CHANGE",
    "BREATHING_CONCERN",
    "HEART_RATE_OR_BP_CONCERN",
    "SKIN_RASH_OR_ALLERGY_CONCERN",
    "MISSED_OR_DELAYED_DOSE",
    "MEDICATION_TIMING_CONTEXT_NEEDED",
]

CARE_PLAN_FOCUS_CODES = [
    "MOBILITY_POSITIONING_SUPPORT",
    "SKIN_PRESSURE_PREVENTION",
    "MEDICATION_ADHERENCE",
    "BOWEL_BLADDER_ROUTINE",
    "RESPIRATORY_MONITORING",
    "REHAB_THERAPY_ROUTINE",
    "FALL_PREVENTION",
    "HYDRATION_NUTRITION",
    "RESPONSIVENESS_MONITORING",
    "PROVIDER_FOLLOW_UP",
    "SEIZURE_REPORTING_SUPPORT",
]

PRIORITY_TYPES = [
    "recurring_concern",
    "emerging_pattern",
    "blind_spot",
    "provider_review_support",
]

ROUTES = [
    "UC4_ROUTINE_CHECKLIST",
    "UC4_PROVIDER_SUMMARY_SUPPORT",
    "UC4_PAUSED_DUE_TO_UC1_EMERGENCY",
]

CONTROLLED_VOCABULARIES = {
    "schema_version": SCHEMA_VERSION,
    "observation_codes": OBSERVATION_CODES,
    "context_codes": CONTEXT_CODES,
    "medication_watch_area_codes": MEDICATION_WATCH_AREA_CODES,
    "care_plan_focus_codes": CARE_PLAN_FOCUS_CODES,
    "priority_types": PRIORITY_TYPES,
    "routes": ROUTES,
}

write_json(OUTPUT_DIR / "uc4_controlled_vocabularies.json", CONTROLLED_VOCABULARIES)

In [9]:
OPTION_SETS = {
    "yes_no_not_sure": [
        {"value": "YES", "label": "Yes"},
        {"value": "NO", "label": "No"},
        {"value": "NOT_SURE", "label": "Not sure"},
    ],
    "timing_relative_to_medication": [
        {"value": "BEFORE_MEDICATION", "label": "Before medication"},
        {"value": "WITHIN_2_HOURS_AFTER", "label": "Within 2 hours after"},
        {"value": "LATER_IN_DAY", "label": "Later in the day"},
        {"value": "NOT_SURE", "label": "Not sure"},
    ],
    "transfer_phase": [
        {"value": "BEFORE_TRANSFER", "label": "Before transfer"},
        {"value": "DURING_TRANSFER", "label": "During transfer"},
        {"value": "AFTER_TRANSFER", "label": "After transfer"},
        {"value": "NOT_SURE", "label": "Not sure"},
    ],
    "position": [
        {"value": "BED", "label": "Bed"},
        {"value": "CHAIR_OR_WHEELCHAIR", "label": "Chair or wheelchair"},
        {"value": "STANDING_OR_WALKING", "label": "Standing or walking"},
        {"value": "OTHER", "label": "Other"},
        {"value": "NOT_SURE", "label": "Not sure"},
    ],
    "discomfort_cues": [
        {"value": "FACIAL_GRIMACE", "label": "Facial grimace"},
        {"value": "GUARDING_OR_STIFFNESS", "label": "Guarding or stiffness"},
        {"value": "VERBAL_DISCOMFORT", "label": "Says they are uncomfortable"},
        {"value": "WITHDRAWAL_OR_IRRITABILITY", "label": "Withdrawal or irritability"},
        {"value": "NONE_OBSERVED", "label": "None observed"},
        {"value": "NOT_SURE", "label": "Not sure"},
    ],
    "breathing_context": [
        {"value": "AT_REST", "label": "At rest"},
        {"value": "AFTER_ACTIVITY", "label": "After activity"},
        {"value": "DURING_SLEEP_OR_NIGHT", "label": "During sleep or at night"},
        {"value": "AROUND_MEDICATION_TIME", "label": "Around medication time"},
        {"value": "NOT_SURE", "label": "Not sure"},
    ],
    "hydration_intake": [
        {"value": "USUAL", "label": "Usual"},
        {"value": "LESS_THAN_USUAL", "label": "Less than usual"},
        {"value": "MUCH_LESS_THAN_USUAL", "label": "Much less than usual"},
        {"value": "NOT_SURE", "label": "Not sure"},
    ],
    "skin_area": [
        {"value": "TAILBONE_OR_SACRUM", "label": "Tailbone/sacrum"},
        {"value": "HIPS", "label": "Hips"},
        {"value": "HEEL_OR_ANKLE", "label": "Heel or ankle"},
        {"value": "BACK", "label": "Back"},
        {"value": "OTHER", "label": "Other"},
        {"value": "NOT_SURE", "label": "Not sure"},
    ],
    "responsiveness_change": [
        {"value": "USUAL", "label": "Usual"},
        {"value": "SLOWER_TO_RESPOND", "label": "Slower to respond"},
        {"value": "MORE_SLEEPY_THAN_USUAL", "label": "More sleepy than usual"},
        {"value": "CONFUSED_OR_NOT_ACTING_USUAL", "label": "Not acting usual"},
        {"value": "NOT_SURE", "label": "Not sure"},
    ],
    "provider_review_reason": [
        {"value": "REPEATED_PATTERN", "label": "Repeated pattern"},
        {"value": "CAREGIVER_CONCERN", "label": "Caregiver concern"},
        {"value": "UPCOMING_VISIT", "label": "Prepare for upcoming visit"},
        {"value": "NOT_SURE", "label": "Not sure"},
    ],
}

write_json(OUTPUT_DIR / "uc4_option_sets.json", OPTION_SETS)

In [11]:
def select_field(field_id: str, label: str, options_key: str, required: bool = True) -> Dict[str, Any]:
    return {
        "field_id": field_id,
        "label": label,
        "input_type": "select",
        "required": required,
        "options_key": options_key,
        "options": OPTION_SETS[options_key],
    }

def multiselect_field(field_id: str, label: str, options_key: str, required: bool = True) -> Dict[str, Any]:
    return {
        "field_id": field_id,
        "label": label,
        "input_type": "multiselect",
        "required": required,
        "options_key": options_key,
        "options": OPTION_SETS[options_key],
    }

def boolean_field(field_id: str, label: str, required: bool = True) -> Dict[str, Any]:
    return {
        "field_id": field_id,
        "label": label,
        "input_type": "boolean",
        "required": required,
    }

def number_field(field_id: str, label: str, required: bool = False, unit: Optional[str] = None) -> Dict[str, Any]:
    field = {
        "field_id": field_id,
        "label": label,
        "input_type": "number",
        "required": required,
    }
    if unit:
        field["unit"] = unit
    return field

In [13]:
TEMPLATE_REGISTRY = {
    "SKIN_PRESSURE_AFTER_SEATED_PERIOD": {
        "template_id": "SKIN_PRESSURE_AFTER_SEATED_PERIOD",
        "template_version": "v0.1.0",
        "title": "Skin and pressure check after seated time",
        "priority_type": "blind_spot",
        "caregiver_summary_template": (
            "{patient_first_name}'s care profile includes skin and pressure prevention as a watch area. "
            "This week, check whether any skin or pressure concerns appear after longer seated or positioned periods."
        ),
        "provider_summary_template": (
            "Structured UC4 priority: skin/pressure monitoring after seated or positioned periods. "
            "Fired rules: {fired_rule_codes}. Evidence: {evidence_summary}."
        ),
        "what_to_log_next_schema": [
            select_field("position_before_check", "Where were they positioned before the check?", "position"),
            select_field("skin_area_checked", "Which area did you check?", "skin_area"),
            select_field("skin_or_pressure_concern_seen", "Did you notice a skin or pressure concern?", "yes_no_not_sure"),
            select_field("repositioning_helped", "Did repositioning seem to help?", "yes_no_not_sure", required=False),
        ],
        "safety_flags": {
            "diagnosis": False,
            "treatment_recommendation": False,
            "medication_causality": False,
            "emergency_override": False,
            "slm_scoring": False,
            "free_text_scoring": False,
        },
    },

    "MEDICATION_WINDOW_FATIGUE_TRACKING": {
        "template_id": "MEDICATION_WINDOW_FATIGUE_TRACKING",
        "template_version": "v0.1.0",
        "title": "Fatigue or sleepiness around medication timing",
        "priority_type": "recurring_concern",
        "caregiver_summary_template": (
            "{patient_first_name}'s medication profile includes fatigue or sleepiness as a known watch area. "
            "This does not mean a medication caused the fatigue. This week, track when fatigue appears relative to medication timing."
        ),
        "provider_summary_template": (
            "Structured UC4 priority: fatigue/sleepiness timing context around medication windows. "
            "No medication causality inferred. Fired rules: {fired_rule_codes}. Evidence: {evidence_summary}."
        ),
        "what_to_log_next_schema": [
            select_field("fatigue_present", "Did fatigue or sleepiness seem unusual?", "yes_no_not_sure"),
            select_field("timing_relative_to_medication", "When did it happen relative to medication timing?", "timing_relative_to_medication"),
            select_field("hydration_or_appetite_change", "Any appetite or hydration change today?", "hydration_intake", required=False),
        ],
        "safety_flags": {
            "diagnosis": False,
            "treatment_recommendation": False,
            "medication_causality": False,
            "emergency_override": False,
            "slm_scoring": False,
            "free_text_scoring": False,
        },
    },

    "MISSED_DELAYED_MEDICATION_CONTEXT": {
        "template_id": "MISSED_DELAYED_MEDICATION_CONTEXT",
        "template_version": "v0.1.0",
        "title": "Missed or delayed medication context",
        "priority_type": "emerging_pattern",
        "caregiver_summary_template": (
            "Recent logs suggest medication timing may be worth tracking for {patient_first_name}. "
            "This is for context only and does not recommend medication changes."
        ),
        "provider_summary_template": (
            "Structured UC4 priority: missed/delayed medication context. No medication change recommended. "
            "Fired rules: {fired_rule_codes}. Evidence: {evidence_summary}."
        ),
        "what_to_log_next_schema": [
            select_field("dose_missed_or_delayed", "Was a dose missed or delayed?", "yes_no_not_sure"),
            select_field("timing_relative_to_medication", "When was the concern noticed?", "timing_relative_to_medication"),
            select_field("caregiver_wants_provider_review", "Would you like this included for provider review?", "yes_no_not_sure", required=False),
        ],
        "safety_flags": {
            "diagnosis": False,
            "treatment_recommendation": False,
            "medication_causality": False,
            "emergency_override": False,
            "slm_scoring": False,
            "free_text_scoring": False,
        },
    },

    "TRANSFER_DISCOMFORT_TRACKING": {
        "template_id": "TRANSFER_DISCOMFORT_TRACKING",
        "template_version": "v0.1.0",
        "title": "Discomfort during transfers or positioning",
        "priority_type": "recurring_concern",
        "caregiver_summary_template": (
            "Recent observations suggest it may be useful to track whether {patient_first_name} shows discomfort during transfers or positioning."
        ),
        "provider_summary_template": (
            "Structured UC4 priority: transfer/positioning discomfort context. Fired rules: {fired_rule_codes}. Evidence: {evidence_summary}."
        ),
        "what_to_log_next_schema": [
            select_field("transfer_phase", "When did discomfort appear?", "transfer_phase"),
            select_field("position_before", "Position before the transfer or repositioning?", "position"),
            select_field("position_after", "Position after the transfer or repositioning?", "position"),
            multiselect_field("discomfort_cues", "What discomfort cues did you notice?", "discomfort_cues"),
            select_field("repositioning_helped", "Did repositioning seem to help?", "yes_no_not_sure", required=False),
        ],
        "safety_flags": {
            "diagnosis": False,
            "treatment_recommendation": False,
            "medication_causality": False,
            "emergency_override": False,
            "slm_scoring": False,
            "free_text_scoring": False,
        },
    },

    "BOWEL_ROUTINE_DISCOMFORT_CONTEXT": {
        "template_id": "BOWEL_ROUTINE_DISCOMFORT_CONTEXT",
        "template_version": "v0.1.0",
        "title": "Bowel/bladder routine and discomfort context",
        "priority_type": "recurring_concern",
        "caregiver_summary_template": (
            "{patient_first_name}'s care profile includes bowel or bladder routine as a watch area. "
            "This week, track whether discomfort, appetite, hydration, or routine timing changed."
        ),
        "provider_summary_template": (
            "Structured UC4 priority: bowel/bladder routine context. Fired rules: {fired_rule_codes}. Evidence: {evidence_summary}."
        ),
        "what_to_log_next_schema": [
            select_field("bowel_bladder_change_seen", "Any bowel or bladder routine change?", "yes_no_not_sure"),
            select_field("hydration_or_appetite_change", "Any appetite or hydration change?", "hydration_intake"),
            select_field("discomfort_present", "Any discomfort noticed?", "yes_no_not_sure"),
        ],
        "safety_flags": {
            "diagnosis": False,
            "treatment_recommendation": False,
            "medication_causality": False,
            "emergency_override": False,
            "slm_scoring": False,
            "free_text_scoring": False,
        },
    },

    "BREATHING_CONCERN_CONTEXT": {
        "template_id": "BREATHING_CONCERN_CONTEXT",
        "template_version": "v0.1.0",
        "title": "Breathing concern context",
        "priority_type": "emerging_pattern",
        "caregiver_summary_template": (
            "{patient_first_name}'s care profile includes breathing as an important watch area. "
            "If this seems urgent or severe, follow the emergency plan. Otherwise, track when the concern appears."
        ),
        "provider_summary_template": (
            "Structured UC4 priority: breathing concern context. Emergency workflow remains separate. "
            "Fired rules: {fired_rule_codes}. Evidence: {evidence_summary}."
        ),
        "what_to_log_next_schema": [
            select_field("breathing_concern_present", "Did you notice a breathing concern?", "yes_no_not_sure"),
            select_field("breathing_context", "When did it happen?", "breathing_context"),
            select_field("caregiver_wants_provider_review", "Would you like this included for provider review?", "yes_no_not_sure", required=False),
        ],
        "safety_flags": {
            "diagnosis": False,
            "treatment_recommendation": False,
            "medication_causality": False,
            "emergency_override": False,
            "slm_scoring": False,
            "free_text_scoring": False,
        },
    },

    "UNUSUAL_RESPONSIVENESS_CONTEXT": {
        "template_id": "UNUSUAL_RESPONSIVENESS_CONTEXT",
        "template_version": "v0.1.0",
        "title": "Responsiveness or alertness context",
        "priority_type": "emerging_pattern",
        "caregiver_summary_template": (
            "{patient_first_name}'s care profile includes responsiveness as a watch area. "
            "If this seems urgent or severe, follow the emergency plan. Otherwise, track how it compares with their usual baseline."
        ),
        "provider_summary_template": (
            "Structured UC4 priority: unusual responsiveness context. Fired rules: {fired_rule_codes}. Evidence: {evidence_summary}."
        ),
        "what_to_log_next_schema": [
            select_field("responsiveness_change", "How did responsiveness compare with usual?", "responsiveness_change"),
            select_field("timing_relative_to_medication", "Did it happen near medication timing?", "timing_relative_to_medication", required=False),
            select_field("caregiver_wants_provider_review", "Would you like this included for provider review?", "yes_no_not_sure", required=False),
        ],
        "safety_flags": {
            "diagnosis": False,
            "treatment_recommendation": False,
            "medication_causality": False,
            "emergency_override": False,
            "slm_scoring": False,
            "free_text_scoring": False,
        },
    },

    "CAREGIVER_REPORTED_SEIZURE_LIKE_EVENT_CONTEXT": {
        "template_id": "CAREGIVER_REPORTED_SEIZURE_LIKE_EVENT_CONTEXT",
        "template_version": "v0.1.0",
        "title": "Caregiver-reported seizure-like event context",
        "priority_type": "provider_review_support",
        "caregiver_summary_template": (
            "A caregiver-reported seizure-like event can be useful to document for the care team. "
            "The app does not detect or diagnose seizures. It only helps organize what you observed."
        ),
        "provider_summary_template": (
            "Structured UC4 priority: caregiver-reported seizure-like event context. No seizure detection or diagnosis performed. "
            "Fired rules: {fired_rule_codes}. Evidence: {evidence_summary}."
        ),
        "what_to_log_next_schema": [
            select_field("event_reported_by_caregiver", "Was an event reported by a caregiver?", "yes_no_not_sure"),
            select_field("responsiveness_change", "How did responsiveness compare with usual?", "responsiveness_change"),
            select_field("caregiver_wants_provider_review", "Include this for provider review?", "yes_no_not_sure"),
        ],
        "safety_flags": {
            "diagnosis": False,
            "treatment_recommendation": False,
            "medication_causality": False,
            "emergency_override": False,
            "slm_scoring": False,
            "free_text_scoring": False,
        },
    },

    "THERAPY_REHAB_ROUTINE_DIFFICULTY": {
        "template_id": "THERAPY_REHAB_ROUTINE_DIFFICULTY",
        "template_version": "v0.1.0",
        "title": "Therapy or rehab routine difficulty",
        "priority_type": "recurring_concern",
        "caregiver_summary_template": (
            "{patient_first_name}'s care plan includes therapy or rehab routines. "
            "This week, track whether difficulty happens before, during, or after the routine."
        ),
        "provider_summary_template": (
            "Structured UC4 priority: therapy/rehab routine difficulty context. Fired rules: {fired_rule_codes}. Evidence: {evidence_summary}."
        ),
        "what_to_log_next_schema": [
            select_field("therapy_difficulty_seen", "Was therapy or rehab more difficult than usual?", "yes_no_not_sure"),
            select_field("therapy_context", "When did the difficulty happen?", "transfer_phase"),
            select_field("discomfort_present", "Any discomfort noticed?", "yes_no_not_sure", required=False),
        ],
        "safety_flags": {
            "diagnosis": False,
            "treatment_recommendation": False,
            "medication_causality": False,
            "emergency_override": False,
            "slm_scoring": False,
            "free_text_scoring": False,
        },
    },

    "FALL_OR_NEAR_FALL_CONTEXT": {
        "template_id": "FALL_OR_NEAR_FALL_CONTEXT",
        "template_version": "v0.1.0",
        "title": "Fall or near-fall context",
        "priority_type": "provider_review_support",
        "caregiver_summary_template": (
            "A fall or near-fall can be important context for {patient_first_name}'s care team. "
            "Use this checklist to document what happened."
        ),
        "provider_summary_template": (
            "Structured UC4 priority: fall or near-fall context. Fired rules: {fired_rule_codes}. Evidence: {evidence_summary}."
        ),
        "what_to_log_next_schema": [
            select_field("fall_or_near_fall_seen", "Was there a fall or near-fall?", "yes_no_not_sure"),
            select_field("position_before", "What was happening before it occurred?", "position"),
            select_field("caregiver_wants_provider_review", "Include this for provider review?", "yes_no_not_sure"),
        ],
        "safety_flags": {
            "diagnosis": False,
            "treatment_recommendation": False,
            "medication_causality": False,
            "emergency_override": False,
            "slm_scoring": False,
            "free_text_scoring": False,
        },
    },

    "CAREGIVER_PROVIDER_REVIEW_REQUEST": {
        "template_id": "CAREGIVER_PROVIDER_REVIEW_REQUEST",
        "template_version": "v0.1.0",
        "title": "Prepare a provider review summary",
        "priority_type": "provider_review_support",
        "caregiver_summary_template": (
            "Recent structured logs suggest it may be useful to prepare a short provider summary for {patient_first_name}."
        ),
        "provider_summary_template": (
            "Structured UC4 provider review support priority. Fired rules: {fired_rule_codes}. Evidence: {evidence_summary}."
        ),
        "what_to_log_next_schema": [
            select_field("provider_review_reason", "Why should this be included for provider review?", "provider_review_reason"),
            select_field("caregiver_wants_provider_review", "Do you want this sent or saved for review?", "yes_no_not_sure"),
        ],
        "safety_flags": {
            "diagnosis": False,
            "treatment_recommendation": False,
            "medication_causality": False,
            "emergency_override": False,
            "slm_scoring": False,
            "free_text_scoring": False,
        },
    },
}

write_json(OUTPUT_DIR / "uc4_template_registry.json", TEMPLATE_REGISTRY)

In [15]:
SYNTHETIC_PATIENT_PROFILES = {
    "Mike_DEMO_001": {
        "patient_id": "Mike_DEMO_001",
        "synthetic": True,
        "first_name": "Mike",
        "profile_label": "cerebral_palsy_high_support_demo",
        "care_plan_focus_areas": [
            "MOBILITY_POSITIONING_SUPPORT",
            "SKIN_PRESSURE_PREVENTION",
            "MEDICATION_ADHERENCE",
            "BOWEL_BLADDER_ROUTINE",
            "SEIZURE_REPORTING_SUPPORT",
            "PROVIDER_FOLLOW_UP",
        ],
        "activated_templates": [
            "SKIN_PRESSURE_AFTER_SEATED_PERIOD",
            "MEDICATION_WINDOW_FATIGUE_TRACKING",
            "MISSED_DELAYED_MEDICATION_CONTEXT",
            "TRANSFER_DISCOMFORT_TRACKING",
            "BOWEL_ROUTINE_DISCOMFORT_CONTEXT",
            "CAREGIVER_REPORTED_SEIZURE_LIKE_EVENT_CONTEXT",
            "CAREGIVER_PROVIDER_REVIEW_REQUEST",
        ],
    },
    "JAMES_DEMO_001": {
        "patient_id": "JAMES_DEMO_001",
        "synthetic": True,
        "first_name": "James",
        "profile_label": "post_stroke_rehab_demo",
        "care_plan_focus_areas": [
            "REHAB_THERAPY_ROUTINE",
            "FALL_PREVENTION",
            "MOBILITY_POSITIONING_SUPPORT",
            "PROVIDER_FOLLOW_UP",
        ],
        "activated_templates": [
            "THERAPY_REHAB_ROUTINE_DIFFICULTY",
            "FALL_OR_NEAR_FALL_CONTEXT",
            "TRANSFER_DISCOMFORT_TRACKING",
            "CAREGIVER_PROVIDER_REVIEW_REQUEST",
        ],
    },
    "SOFIA_DEMO_001": {
        "patient_id": "SOFIA_DEMO_001",
        "synthetic": True,
        "first_name": "Sofia",
        "profile_label": "spina_bifida_demo",
        "care_plan_focus_areas": [
            "BOWEL_BLADDER_ROUTINE",
            "HYDRATION_NUTRITION",
            "SKIN_PRESSURE_PREVENTION",
            "MEDICATION_ADHERENCE",
            "PROVIDER_FOLLOW_UP",
        ],
        "activated_templates": [
            "BOWEL_ROUTINE_DISCOMFORT_CONTEXT",
            "SKIN_PRESSURE_AFTER_SEATED_PERIOD",
            "MISSED_DELAYED_MEDICATION_CONTEXT",
            "MEDICATION_WINDOW_FATIGUE_TRACKING",
            "CAREGIVER_PROVIDER_REVIEW_REQUEST",
        ],
    },
    "ELENA_DEMO_001": {
        "patient_id": "ELENA_DEMO_001",
        "synthetic": True,
        "first_name": "Elena",
        "profile_label": "copd_tbi_demo",
        "care_plan_focus_areas": [
            "RESPIRATORY_MONITORING",
            "RESPONSIVENESS_MONITORING",
            "MEDICATION_ADHERENCE",
            "PROVIDER_FOLLOW_UP",
        ],
        "activated_templates": [
            "BREATHING_CONCERN_CONTEXT",
            "UNUSUAL_RESPONSIVENESS_CONTEXT",
            "MEDICATION_WINDOW_FATIGUE_TRACKING",
            "CAREGIVER_PROVIDER_REVIEW_REQUEST",
        ],
    },
}

write_json(OUTPUT_DIR / "synthetic_patient_profiles.json", SYNTHETIC_PATIENT_PROFILES)

In [17]:
SYNTHETIC_MEDICATION_PROFILES = {
    "Mike_DEMO_001": {
        "patient_id": "Mike_DEMO_001",
        "synthetic": True,
        "medications": [
            {
                "name": "Synthetic antispasticity medication",
                "watch_areas": ["SLEEPINESS_FATIGUE", "WEAKNESS_OR_LOW_TONE_CONCERN"],
            },
            {
                "name": "Synthetic bowel regimen medication",
                "watch_areas": ["BOWEL_CHANGE", "APPETITE_OR_HYDRATION_CHANGE"],
            },
        ],
    },
    "JAMES_DEMO_001": {
        "patient_id": "JAMES_DEMO_001",
        "synthetic": True,
        "medications": [
            {
                "name": "Synthetic blood pressure medication",
                "watch_areas": ["DIZZINESS_OR_LIGHTHEADEDNESS", "HEART_RATE_OR_BP_CONCERN"],
            }
        ],
    },
    "SOFIA_DEMO_001": {
        "patient_id": "SOFIA_DEMO_001",
        "synthetic": True,
        "medications": [
            {
                "name": "Synthetic bladder/bowel support medication",
                "watch_areas": ["BOWEL_CHANGE", "APPETITE_OR_HYDRATION_CHANGE"],
            }
        ],
    },
    "ELENA_DEMO_001": {
        "patient_id": "ELENA_DEMO_001",
        "synthetic": True,
        "medications": [
            {
                "name": "Synthetic respiratory medication",
                "watch_areas": ["BREATHING_CONCERN", "SLEEPINESS_FATIGUE"],
            }
        ],
    },
}

def med_watch_areas_for_patient(patient_id: str) -> List[str]:
    profile = SYNTHETIC_MEDICATION_PROFILES.get(patient_id, {})
    areas = []
    for med in profile.get("medications", []):
        areas.extend(med.get("watch_areas", []))
    return sorted(set(areas))

write_json(OUTPUT_DIR / "synthetic_medication_profiles.json", SYNTHETIC_MEDICATION_PROFILES)

med_rows = []
for pid, prof in SYNTHETIC_MEDICATION_PROFILES.items():
    for med in prof["medications"]:
        med_rows.append({
            "patient_id": pid,
            "synthetic": True,
            "medication_name": med["name"],
            "watch_areas": "|".join(med["watch_areas"]),
        })
pd.DataFrame(med_rows).to_csv(OUTPUT_DIR / "synthetic_medication_profiles.csv", index=False)

In [19]:
SYNTHETIC_SHARED_EVENTS = [
    # Mike
    {
        "patient_id": "Mike_DEMO_001",
        "source": "caregiver_log",
        "event_type": "STRUCTURED_OBSERVATION",
        "days_ago": 1,
        "observation_codes": ["UNUSUAL_FATIGUE", "TRANSFER_OR_POSITIONING_CONTEXT", "PAIN_OR_DISCOMFORT"],
        "context_codes": ["DURING_TRANSFER", "AROUND_MEDICATION_TIME"],
        "severity": 1,
        "free_text_used_for_scoring": False,
    },
    {
        "patient_id": "Mike_DEMO_001",
        "source": "caregiver_log",
        "event_type": "STRUCTURED_OBSERVATION",
        "days_ago": 3,
        "observation_codes": ["UNUSUAL_FATIGUE", "BOWEL_OR_BLADDER_CHANGE"],
        "context_codes": ["BATHROOM_OR_BOWEL_BLADDER"],
        "severity": 1,
        "free_text_used_for_scoring": False,
    },
    {
        "patient_id": "Mike_DEMO_001",
        "source": "caregiver_log",
        "event_type": "STRUCTURED_OBSERVATION",
        "days_ago": 4,
        "observation_codes": ["SKIN_OR_PRESSURE_CONCERN", "LOW_MOVEMENT"],
        "context_codes": ["WHILE_SITTING_OR_POSITIONED"],
        "severity": 1,
        "free_text_used_for_scoring": False,
    },
    {
        "patient_id": "Mike_DEMO_001",
        "source": "caregiver_log",
        "event_type": "STRUCTURED_OBSERVATION",
        "days_ago": 5,
        "observation_codes": ["SEIZURE_LIKE_EVENT_REPORTED", "UNUSUAL_RESPONSIVENESS"],
        "context_codes": ["UNKNOWN_OR_NOT_SURE"],
        "severity": 2,
        "free_text_used_for_scoring": False,
    },

    # James
    {
        "patient_id": "JAMES_DEMO_001",
        "source": "caregiver_log",
        "event_type": "STRUCTURED_OBSERVATION",
        "days_ago": 1,
        "observation_codes": ["THERAPY_ROUTINE_DIFFICULTY", "PAIN_OR_DISCOMFORT"],
        "context_codes": ["AFTER_ACTIVITY_OR_THERAPY"],
        "severity": 1,
        "free_text_used_for_scoring": False,
    },
    {
        "patient_id": "JAMES_DEMO_001",
        "source": "caregiver_log",
        "event_type": "STRUCTURED_OBSERVATION",
        "days_ago": 2,
        "observation_codes": ["FALL_OR_NEAR_FALL", "CAREGIVER_WANTS_PROVIDER_REVIEW"],
        "context_codes": ["DURING_TRANSFER"],
        "severity": 2,
        "free_text_used_for_scoring": False,
    },
    {
        "patient_id": "JAMES_DEMO_001",
        "source": "UC3_TRAJECTORY",
        "event_type": "TRAJECTORY_FAILURE_DETECTED",
        "days_ago": 3,
        "observation_codes": ["THERAPY_ROUTINE_DIFFICULTY"],
        "context_codes": ["AFTER_ACTIVITY_OR_THERAPY"],
        "severity": 2,
        "free_text_used_for_scoring": False,
    },

    # Sofia
    {
        "patient_id": "SOFIA_DEMO_001",
        "source": "UC2_SLOW_PATH_ANOMALY",
        "event_type": "TRIGGER_WORKFLOW_ANOMALY_TYPE_04",
        "days_ago": 1,
        "observation_codes": ["UNUSUAL_FATIGUE", "APPETITE_OR_HYDRATION_CHANGE"],
        "context_codes": ["MEAL_OR_HYDRATION_RELATED"],
        "severity": 2,
        "free_text_used_for_scoring": False,
    },
    {
        "patient_id": "SOFIA_DEMO_001",
        "source": "caregiver_log",
        "event_type": "STRUCTURED_OBSERVATION",
        "days_ago": 2,
        "observation_codes": ["BOWEL_OR_BLADDER_CHANGE"],
        "context_codes": ["BATHROOM_OR_BOWEL_BLADDER"],
        "severity": 1,
        "free_text_used_for_scoring": False,
    },
    {
        "patient_id": "SOFIA_DEMO_001",
        "source": "caregiver_log",
        "event_type": "STRUCTURED_OBSERVATION",
        "days_ago": 6,
        "observation_codes": ["MISSED_OR_DELAYED_MEDICATION"],
        "context_codes": ["AROUND_MEDICATION_TIME"],
        "severity": 1,
        "free_text_used_for_scoring": False,
    },

    # Elena
    {
        "patient_id": "ELENA_DEMO_001",
        "source": "caregiver_log",
        "event_type": "STRUCTURED_OBSERVATION",
        "days_ago": 1,
        "observation_codes": ["BREATHING_CONCERN", "UNUSUAL_FATIGUE"],
        "context_codes": ["AFTER_ACTIVITY_OR_THERAPY"],
        "severity": 1,
        "free_text_used_for_scoring": False,
    },
    {
        "patient_id": "ELENA_DEMO_001",
        "source": "caregiver_log",
        "event_type": "STRUCTURED_OBSERVATION",
        "days_ago": 2,
        "observation_codes": ["UNUSUAL_RESPONSIVENESS"],
        "context_codes": ["DURING_SLEEP_OR_NIGHT"],
        "severity": 1,
        "free_text_used_for_scoring": False,
    },
    {
        "patient_id": "ELENA_DEMO_001",
        "source": "UC1_ACUTE_EMERGENCY",
        "event_type": "SEVERITY_3_EMERGENCY_RESOLVED",
        "days_ago": 5,
        "observation_codes": ["BREATHING_CONCERN", "COLOR_OR_OXYGEN_CONCERN"],
        "context_codes": ["UNKNOWN_OR_NOT_SURE"],
        "severity": 3,
        "free_text_used_for_scoring": False,
    },
]

events_df = pd.DataFrame(SYNTHETIC_SHARED_EVENTS)
events_df.to_csv(OUTPUT_DIR / "synthetic_shared_care_events.csv", index=False)
events_df

,patient_id,source,event_type,days_ago,observation_codes,context_codes,severity,free_text_used_for_scoring
0,Mike_DEMO_001,caregiver_log,STRUCTURED_OBSERVATION,1,"[UNUSUAL_FATIGUE, TRANSFER_OR_POSITIONING_CONT...","[DURING_TRANSFER, AROUND_MEDICATION_TIME]",1,False
1,Mike_DEMO_001,caregiver_log,STRUCTURED_OBSERVATION,3,"[UNUSUAL_FATIGUE, BOWEL_OR_BLADDER_CHANGE]",[BATHROOM_OR_BOWEL_BLADDER],1,False
2,Mike_DEMO_001,caregiver_log,STRUCTURED_OBSERVATION,4,"[SKIN_OR_PRESSURE_CONCERN, LOW_MOVEMENT]",[WHILE_SITTING_OR_POSITIONED],1,False
3,Mike_DEMO_001,caregiver_log,STRUCTURED_OBSERVATION,5,"[SEIZURE_LIKE_EVENT_REPORTED, UNUSUAL_RESPONSI...",[UNKNOWN_OR_NOT_SURE],2,False
4,JAMES_DEMO_001,caregiver_log,STRUCTURED_OBSERVATION,1,"[THERAPY_ROUTINE_DIFFICULTY, PAIN_OR_DISCOMFORT]",[AFTER_ACTIVITY_OR_THERAPY],1,False
5,JAMES_DEMO_001,caregiver_log,STRUCTURED_OBSERVATION,2,"[FALL_OR_NEAR_FALL, CAREGIVER_WANTS_PROVIDER_R...",[DURING_TRANSFER],2,False
6,JAMES_DEMO_001,UC3_TRAJECTORY,TRAJECTORY_FAILURE_DETECTED,3,[THERAPY_ROUTINE_DIFFICULTY],[AFTER_ACTIVITY_OR_THERAPY],2,False
7,SOFIA_DEMO_001,UC2_SLOW_PATH_ANOMALY,TRIGGER_WORKFLOW_ANOMALY_TYPE_04,1,"[UNUSUAL_FATIGUE, APPETITE_OR_HYDRATION_CHANGE]",[MEAL_OR_HYDRATION_RELATED],2,False
8,SOFIA_DEMO_001,caregiver_log,STRUCTURED_OBSERVATION,2,[BOWEL_OR_BLADDER_CHANGE],[BATHROOM_OR_BOWEL_BLADDER],1,False
9,SOFIA_DEMO_001,caregiver_log,STRUCTURED_OBSERVATION,6,[MISSED_OR_DELAYED_MEDICATION],[AROUND_MEDICATION_TIME],1,False


In [21]:
SYNTHETIC_WEARABLE_WEEKLY_SUMMARIES = {
    "Mike_DEMO_001": {
        "patient_id": "Mike_DEMO_001",
        "synthetic": True,
        "low_movement_delta_7d": 2.3,
        "sleep_disruption_delta_7d": 1.1,
        "respiratory_concern_delta_7d": 0.0,
    },
    "JAMES_DEMO_001": {
        "patient_id": "JAMES_DEMO_001",
        "synthetic": True,
        "low_movement_delta_7d": 1.5,
        "sleep_disruption_delta_7d": 0.3,
        "respiratory_concern_delta_7d": 0.0,
    },
    "SOFIA_DEMO_001": {
        "patient_id": "SOFIA_DEMO_001",
        "synthetic": True,
        "low_movement_delta_7d": 1.8,
        "sleep_disruption_delta_7d": 0.8,
        "respiratory_concern_delta_7d": 0.4,
    },
    "ELENA_DEMO_001": {
        "patient_id": "ELENA_DEMO_001",
        "synthetic": True,
        "low_movement_delta_7d": 1.0,
        "sleep_disruption_delta_7d": 1.7,
        "respiratory_concern_delta_7d": 2.4,
    },
}

write_json(OUTPUT_DIR / "synthetic_wearable_weekly_summaries.json", SYNTHETIC_WEARABLE_WEEKLY_SUMMARIES)

In [23]:
PREVIOUS_UC4_PRIORITIES = {
    "Mike_DEMO_001": {
        "previous_priority_template_ids": ["TRANSFER_DISCOMFORT_TRACKING"],
        "dismissed_template_ids": [],
        "confirmed_useful_template_ids": ["TRANSFER_DISCOMFORT_TRACKING"],
    },
    "JAMES_DEMO_001": {
        "previous_priority_template_ids": [],
        "dismissed_template_ids": [],
        "confirmed_useful_template_ids": [],
    },
    "SOFIA_DEMO_001": {
        "previous_priority_template_ids": ["BOWEL_ROUTINE_DISCOMFORT_CONTEXT"],
        "dismissed_template_ids": [],
        "confirmed_useful_template_ids": ["BOWEL_ROUTINE_DISCOMFORT_CONTEXT"],
    },
    "ELENA_DEMO_001": {
        "previous_priority_template_ids": [],
        "dismissed_template_ids": [],
        "confirmed_useful_template_ids": [],
    },
}

write_json(OUTPUT_DIR / "previous_uc4_priorities.json", PREVIOUS_UC4_PRIORITIES)

In [25]:
def count_codes(events: List[Dict[str, Any]], code_field: str, allowed_codes: List[str], within_days: int) -> Dict[str, int]:
    counts = {code: 0 for code in allowed_codes}
    for ev in events:
        if ev.get("days_ago", 999) <= within_days:
            for code in ev.get(code_field, []):
                if code in counts:
                    counts[code] += 1
    return counts

def build_rule_context(patient_id: str, uc1_emergency_active: bool = False) -> Dict[str, Any]:
    profile = SYNTHETIC_PATIENT_PROFILES[patient_id]
    patient_events = [e for e in SYNTHETIC_SHARED_EVENTS if e["patient_id"] == patient_id]
    wearable = SYNTHETIC_WEARABLE_WEEKLY_SUMMARIES[patient_id]
    previous = PREVIOUS_UC4_PRIORITIES[patient_id]

    observation_counts_7d = count_codes(patient_events, "observation_codes", OBSERVATION_CODES, within_days=7)
    observation_counts_14d = count_codes(patient_events, "observation_codes", OBSERVATION_CODES, within_days=14)
    context_counts_7d = count_codes(patient_events, "context_codes", CONTEXT_CODES, within_days=7)

    uc2_recent_events = [
        e for e in patient_events 
        if e["source"] == "UC2_SLOW_PATH_ANOMALY" and e.get("days_ago", 999) <= 7
    ]
    uc1_severity3_recent = any(
        e["source"] == "UC1_ACUTE_EMERGENCY" and e.get("severity") == 3 and e.get("days_ago", 999) <= 7
        for e in patient_events
    )

    caregiver_provider_review_count_7d = observation_counts_7d.get("CAREGIVER_WANTS_PROVIDER_REVIEW", 0)

    ctx = {
        "schema_version": SCHEMA_VERSION,
        "patient_id": patient_id,
        "patient_first_name": profile["first_name"],
        "synthetic": True,

        "observation_counts_7d": observation_counts_7d,
        "observation_counts_14d": observation_counts_14d,
        "context_counts_7d": context_counts_7d,

        "medication_watch_areas": med_watch_areas_for_patient(patient_id),
        "care_plan_focus_areas": profile["care_plan_focus_areas"],
        "activated_templates": profile["activated_templates"],

        "wearable_deltas": {
            "low_movement_delta_7d": wearable["low_movement_delta_7d"],
            "sleep_disruption_delta_7d": wearable["sleep_disruption_delta_7d"],
            "respiratory_concern_delta_7d": wearable["respiratory_concern_delta_7d"],
        },

        "previous_priority_template_ids": previous["previous_priority_template_ids"],

        "missing_schema_field_counts": {
            # Synthetic examples: these fields were not well captured previously.
            "timing_relative_to_medication": 2 if patient_id in ["Mike_DEMO_001", "SOFIA_DEMO_001"] else 0,
            "position_before": 2 if patient_id == "Mike_DEMO_001" else 0,
            "breathing_context": 2 if patient_id == "ELENA_DEMO_001" else 0,
        },

        "caregiver_preference_signals": {
            "dismissed_template_ids": previous["dismissed_template_ids"],
            "confirmed_useful_template_ids": previous["confirmed_useful_template_ids"],
            "requested_provider_review_count_7d": caregiver_provider_review_count_7d,
        },

        "uc2_recent_events": uc2_recent_events,

        "safety_flags": {
            "uc1_emergency_active": uc1_emergency_active,
            "uc2_severity3_recent": False,
            "uc1_severity3_recent_resolved": uc1_severity3_recent,
        },

        "free_text_used_for_scoring": False,
        "slm_used_for_scoring": False,
    }

    return ctx

RULE_CONTEXTS_BY_PATIENT = {
    pid: build_rule_context(pid)
    for pid in SYNTHETIC_PATIENT_PROFILES.keys()
}

write_json(OUTPUT_DIR / "uc4_rule_contexts_by_patient.json", RULE_CONTEXTS_BY_PATIENT)
RULE_CONTEXTS_BY_PATIENT["Mike_DEMO_001"]

{'schema_version': 'uc4_schema_v0.1.0',
 'patient_id': 'Mike_DEMO_001',
 'patient_first_name': 'Mike',
 'synthetic': True,
 'observation_counts_7d': {'LOOKS_NORMAL': 0,
  'NOT_SURE': 0,
  'DEVICE_OR_SENSOR_ISSUE': 0,
  'RECENT_ACTIVITY_OR_EXERTION': 0,
  'LOW_MOVEMENT': 1,
  'TRANSFER_OR_POSITIONING_CONTEXT': 1,
  'PAIN_OR_DISCOMFORT': 1,
  'UNUSUAL_FATIGUE': 2,
  'POOR_SLEEP_OR_RESTLESSNESS': 0,
  'BREATHING_CONCERN': 0,
  'COLOR_OR_OXYGEN_CONCERN': 0,
  'MISSED_OR_DELAYED_MEDICATION': 0,
  'RECENT_MEDICATION_CHANGE': 0,
  'APPETITE_OR_HYDRATION_CHANGE': 0,
  'BOWEL_OR_BLADDER_CHANGE': 1,
  'SKIN_OR_PRESSURE_CONCERN': 1,
  'SEIZURE_LIKE_EVENT_REPORTED': 1,
  'UNUSUAL_RESPONSIVENESS': 1,
  'CAREGIVER_WANTS_PROVIDER_REVIEW': 0,
  'FALL_OR_NEAR_FALL': 0,
  'THERAPY_ROUTINE_DIFFICULTY': 0},
 'observation_counts_14d': {'LOOKS_NORMAL': 0,
  'NOT_SURE': 0,
  'DEVICE_OR_SENSOR_ISSUE': 0,
  'RECENT_ACTIVITY_OR_EXERTION': 0,
  'LOW_MOVEMENT': 1,
  'TRANSFER_OR_POSITIONING_CONTEXT': 1,
  'PAIN_O

In [27]:
def get_nested_value(obj: Dict[str, Any], path: str) -> Any:
    current = obj
    for part in path.split("."):
        if isinstance(current, dict):
            current = current.get(part)
        else:
            return None
    return current

def evidence_ref(field_path: str, value: Any, comparator: Optional[str] = None, threshold: Any = None) -> Dict[str, Any]:
    ref = {
        "field_path": field_path,
        "value": value,
    }
    if comparator is not None:
        ref["comparator"] = comparator
    if threshold is not None:
        ref["threshold"] = threshold
    return ref

def has_focus(ctx: Dict[str, Any], focus_code: str) -> bool:
    return focus_code in ctx.get("care_plan_focus_areas", [])

def has_med_watch(ctx: Dict[str, Any], watch_code: str) -> bool:
    return watch_code in ctx.get("medication_watch_areas", [])

def obs_count(ctx: Dict[str, Any], code: str, days: int = 7) -> int:
    key = "observation_counts_7d" if days == 7 else "observation_counts_14d"
    return int(ctx.get(key, {}).get(code, 0) or 0)

def context_count(ctx: Dict[str, Any], code: str) -> int:
    return int(ctx.get("context_counts_7d", {}).get(code, 0) or 0)

def wearable_delta(ctx: Dict[str, Any], key: str) -> float:
    return float(ctx.get("wearable_deltas", {}).get(key, 0.0) or 0.0)

def missing_field_count(ctx: Dict[str, Any], field_id: str) -> int:
    return int(ctx.get("missing_schema_field_counts", {}).get(field_id, 0) or 0)

In [30]:
def make_rule(
    rule_code: str,
    description: str,
    weight: float,
    applies_to_templates: List[str],
    evidence_fields: List[str],
    evaluate: Callable[[Dict[str, Any]], bool],
    build_evidence: Callable[[Dict[str, Any]], List[Dict[str, Any]]],
) -> Dict[str, Any]:
    return {
        "rule_code": rule_code,
        "rule_registry_version": RULE_REGISTRY_VERSION,
        "description": description,
        "weight": weight,
        "applies_to_templates": applies_to_templates,
        "evidence_fields": evidence_fields,
        "evaluate": evaluate,
        "build_evidence": build_evidence,
        "safety_tags": {
            "diagnostic": False,
            "treatment_recommendation": False,
            "medication_causality": False,
            "emergency_override": False,
            "slm_scoring": False,
            "free_text_scoring": False,
        },
        "enabled": True,
    }

In [32]:
RULE_REGISTRY = [
    make_rule(
        "R_FATIGUE_RECURRENCE",
        "Unusual fatigue has been logged at least twice in the last 7 days.",
        0.40,
        ["MEDICATION_WINDOW_FATIGUE_TRACKING"],
        ["observation_counts_7d.UNUSUAL_FATIGUE"],
        lambda ctx: obs_count(ctx, "UNUSUAL_FATIGUE") >= 2,
        lambda ctx: [
            evidence_ref(
                "observation_counts_7d.UNUSUAL_FATIGUE",
                obs_count(ctx, "UNUSUAL_FATIGUE"),
                ">=",
                2
            )
        ],
    ),

    make_rule(
        "R_MED_WATCH_AREA_MATCH_FATIGUE",
        "Medication profile includes sleepiness/fatigue as a known watch area.",
        0.30,
        ["MEDICATION_WINDOW_FATIGUE_TRACKING"],
        ["medication_watch_areas"],
        lambda ctx: has_med_watch(ctx, "SLEEPINESS_FATIGUE"),
        lambda ctx: [
            evidence_ref(
                "medication_watch_areas",
                ctx["medication_watch_areas"],
                "includes",
                "SLEEPINESS_FATIGUE"
            )
        ],
    ),

    make_rule(
        "R_MED_TIMING_CONTEXT_MISSING",
        "Medication timing context has been missing in recent logs.",
        0.20,
        ["MEDICATION_WINDOW_FATIGUE_TRACKING", "MISSED_DELAYED_MEDICATION_CONTEXT"],
        ["missing_schema_field_counts.timing_relative_to_medication"],
        lambda ctx: missing_field_count(ctx, "timing_relative_to_medication") >= 2,
        lambda ctx: [
            evidence_ref(
                "missing_schema_field_counts.timing_relative_to_medication",
                missing_field_count(ctx, "timing_relative_to_medication"),
                ">=",
                2
            )
        ],
    ),

    make_rule(
        "R_MISSED_DELAYED_MEDICATION_LOGGED",
        "Missed or delayed medication was logged recently.",
        0.50,
        ["MISSED_DELAYED_MEDICATION_CONTEXT"],
        ["observation_counts_7d.MISSED_OR_DELAYED_MEDICATION"],
        lambda ctx: obs_count(ctx, "MISSED_OR_DELAYED_MEDICATION") >= 1,
        lambda ctx: [
            evidence_ref(
                "observation_counts_7d.MISSED_OR_DELAYED_MEDICATION",
                obs_count(ctx, "MISSED_OR_DELAYED_MEDICATION"),
                ">=",
                1
            )
        ],
    ),

    make_rule(
        "R_TRANSFER_DISCOMFORT_RECURRENCE",
        "Discomfort and transfer/positioning context have both been logged recently.",
        0.55,
        ["TRANSFER_DISCOMFORT_TRACKING"],
        [
            "observation_counts_7d.PAIN_OR_DISCOMFORT",
            "observation_counts_7d.TRANSFER_OR_POSITIONING_CONTEXT",
        ],
        lambda ctx: obs_count(ctx, "PAIN_OR_DISCOMFORT") >= 1 and obs_count(ctx, "TRANSFER_OR_POSITIONING_CONTEXT") >= 1,
        lambda ctx: [
            evidence_ref("observation_counts_7d.PAIN_OR_DISCOMFORT", obs_count(ctx, "PAIN_OR_DISCOMFORT"), ">=", 1),
            evidence_ref("observation_counts_7d.TRANSFER_OR_POSITIONING_CONTEXT", obs_count(ctx, "TRANSFER_OR_POSITIONING_CONTEXT"), ">=", 1),
        ],
    ),

    make_rule(
        "R_TRANSFER_CONTEXT_CLUSTER",
        "Transfer context has appeared in recent logs.",
        0.30,
        ["TRANSFER_DISCOMFORT_TRACKING", "FALL_OR_NEAR_FALL_CONTEXT"],
        ["context_counts_7d.DURING_TRANSFER"],
        lambda ctx: context_count(ctx, "DURING_TRANSFER") >= 1,
        lambda ctx: [
            evidence_ref("context_counts_7d.DURING_TRANSFER", context_count(ctx, "DURING_TRANSFER"), ">=", 1)
        ],
    ),

    make_rule(
        "R_LOW_MOVEMENT_INCREASE",
        "Low movement pattern has increased compared with baseline.",
        0.35,
        ["SKIN_PRESSURE_AFTER_SEATED_PERIOD", "TRANSFER_DISCOMFORT_TRACKING"],
        ["wearable_deltas.low_movement_delta_7d"],
        lambda ctx: wearable_delta(ctx, "low_movement_delta_7d") >= 2.0,
        lambda ctx: [
            evidence_ref("wearable_deltas.low_movement_delta_7d", wearable_delta(ctx, "low_movement_delta_7d"), ">=", 2.0)
        ],
    ),

    make_rule(
        "R_SKIN_PRESSURE_FOCUS",
        "Care plan includes skin/pressure prevention.",
        0.30,
        ["SKIN_PRESSURE_AFTER_SEATED_PERIOD"],
        ["care_plan_focus_areas"],
        lambda ctx: has_focus(ctx, "SKIN_PRESSURE_PREVENTION"),
        lambda ctx: [
            evidence_ref("care_plan_focus_areas", ctx["care_plan_focus_areas"], "includes", "SKIN_PRESSURE_PREVENTION")
        ],
    ),

    make_rule(
        "R_SKIN_OR_PRESSURE_CONCERN_LOGGED",
        "Skin or pressure concern was logged recently.",
        0.45,
        ["SKIN_PRESSURE_AFTER_SEATED_PERIOD"],
        ["observation_counts_7d.SKIN_OR_PRESSURE_CONCERN"],
        lambda ctx: obs_count(ctx, "SKIN_OR_PRESSURE_CONCERN") >= 1,
        lambda ctx: [
            evidence_ref("observation_counts_7d.SKIN_OR_PRESSURE_CONCERN", obs_count(ctx, "SKIN_OR_PRESSURE_CONCERN"), ">=", 1)
        ],
    ),

    make_rule(
        "R_BOWEL_BLADDER_FOCUS",
        "Care plan includes bowel/bladder routine.",
        0.30,
        ["BOWEL_ROUTINE_DISCOMFORT_CONTEXT"],
        ["care_plan_focus_areas"],
        lambda ctx: has_focus(ctx, "BOWEL_BLADDER_ROUTINE"),
        lambda ctx: [
            evidence_ref("care_plan_focus_areas", ctx["care_plan_focus_areas"], "includes", "BOWEL_BLADDER_ROUTINE")
        ],
    ),

    make_rule(
        "R_BOWEL_ROUTINE_CHANGE_LOGGED",
        "Bowel or bladder change has been logged recently.",
        0.45,
        ["BOWEL_ROUTINE_DISCOMFORT_CONTEXT"],
        ["observation_counts_7d.BOWEL_OR_BLADDER_CHANGE"],
        lambda ctx: obs_count(ctx, "BOWEL_OR_BLADDER_CHANGE") >= 1,
        lambda ctx: [
            evidence_ref("observation_counts_7d.BOWEL_OR_BLADDER_CHANGE", obs_count(ctx, "BOWEL_OR_BLADDER_CHANGE"), ">=", 1)
        ],
    ),

    make_rule(
        "R_APPETITE_HYDRATION_CHANGE_LOGGED",
        "Appetite or hydration change has been logged recently.",
        0.30,
        ["BOWEL_ROUTINE_DISCOMFORT_CONTEXT", "MEDICATION_WINDOW_FATIGUE_TRACKING"],
        ["observation_counts_7d.APPETITE_OR_HYDRATION_CHANGE"],
        lambda ctx: obs_count(ctx, "APPETITE_OR_HYDRATION_CHANGE") >= 1,
        lambda ctx: [
            evidence_ref("observation_counts_7d.APPETITE_OR_HYDRATION_CHANGE", obs_count(ctx, "APPETITE_OR_HYDRATION_CHANGE"), ">=", 1)
        ],
    ),

    make_rule(
        "R_BREATHING_FOCUS",
        "Care plan includes respiratory monitoring.",
        0.35,
        ["BREATHING_CONCERN_CONTEXT"],
        ["care_plan_focus_areas"],
        lambda ctx: has_focus(ctx, "RESPIRATORY_MONITORING"),
        lambda ctx: [
            evidence_ref("care_plan_focus_areas", ctx["care_plan_focus_areas"], "includes", "RESPIRATORY_MONITORING")
        ],
    ),

    make_rule(
        "R_BREATHING_CONCERN_LOGGED",
        "Breathing concern has been logged recently.",
        0.55,
        ["BREATHING_CONCERN_CONTEXT"],
        ["observation_counts_7d.BREATHING_CONCERN"],
        lambda ctx: obs_count(ctx, "BREATHING_CONCERN") >= 1,
        lambda ctx: [
            evidence_ref("observation_counts_7d.BREATHING_CONCERN", obs_count(ctx, "BREATHING_CONCERN"), ">=", 1)
        ],
    ),

    make_rule(
        "R_RESPIRATORY_WEARABLE_DELTA",
        "Respiratory concern wearable summary increased compared with baseline.",
        0.30,
        ["BREATHING_CONCERN_CONTEXT"],
        ["wearable_deltas.respiratory_concern_delta_7d"],
        lambda ctx: wearable_delta(ctx, "respiratory_concern_delta_7d") >= 2.0,
        lambda ctx: [
            evidence_ref("wearable_deltas.respiratory_concern_delta_7d", wearable_delta(ctx, "respiratory_concern_delta_7d"), ">=", 2.0)
        ],
    ),

    make_rule(
        "R_RESPONSIVENESS_FOCUS",
        "Care plan includes responsiveness monitoring.",
        0.35,
        ["UNUSUAL_RESPONSIVENESS_CONTEXT"],
        ["care_plan_focus_areas"],
        lambda ctx: has_focus(ctx, "RESPONSIVENESS_MONITORING"),
        lambda ctx: [
            evidence_ref("care_plan_focus_areas", ctx["care_plan_focus_areas"], "includes", "RESPONSIVENESS_MONITORING")
        ],
    ),

    make_rule(
        "R_UNUSUAL_RESPONSIVENESS_LOGGED",
        "Unusual responsiveness was logged recently.",
        0.55,
        ["UNUSUAL_RESPONSIVENESS_CONTEXT", "CAREGIVER_REPORTED_SEIZURE_LIKE_EVENT_CONTEXT"],
        ["observation_counts_7d.UNUSUAL_RESPONSIVENESS"],
        lambda ctx: obs_count(ctx, "UNUSUAL_RESPONSIVENESS") >= 1,
        lambda ctx: [
            evidence_ref("observation_counts_7d.UNUSUAL_RESPONSIVENESS", obs_count(ctx, "UNUSUAL_RESPONSIVENESS"), ">=", 1)
        ],
    ),

    make_rule(
        "R_CAREGIVER_REPORTED_SEIZURE_LIKE_EVENT",
        "Caregiver reported a seizure-like event. App does not detect or diagnose seizures.",
        0.70,
        ["CAREGIVER_REPORTED_SEIZURE_LIKE_EVENT_CONTEXT"],
        ["observation_counts_7d.SEIZURE_LIKE_EVENT_REPORTED"],
        lambda ctx: obs_count(ctx, "SEIZURE_LIKE_EVENT_REPORTED") >= 1,
        lambda ctx: [
            evidence_ref("observation_counts_7d.SEIZURE_LIKE_EVENT_REPORTED", obs_count(ctx, "SEIZURE_LIKE_EVENT_REPORTED"), ">=", 1)
        ],
    ),

    make_rule(
        "R_THERAPY_ROUTINE_DIFFICULTY",
        "Therapy or rehab difficulty has been logged recently.",
        0.60,
        ["THERAPY_REHAB_ROUTINE_DIFFICULTY"],
        ["observation_counts_7d.THERAPY_ROUTINE_DIFFICULTY"],
        lambda ctx: obs_count(ctx, "THERAPY_ROUTINE_DIFFICULTY") >= 1,
        lambda ctx: [
            evidence_ref("observation_counts_7d.THERAPY_ROUTINE_DIFFICULTY", obs_count(ctx, "THERAPY_ROUTINE_DIFFICULTY"), ">=", 1)
        ],
    ),

    make_rule(
        "R_REHAB_FOCUS",
        "Care plan includes therapy or rehab routine.",
        0.30,
        ["THERAPY_REHAB_ROUTINE_DIFFICULTY"],
        ["care_plan_focus_areas"],
        lambda ctx: has_focus(ctx, "REHAB_THERAPY_ROUTINE"),
        lambda ctx: [
            evidence_ref("care_plan_focus_areas", ctx["care_plan_focus_areas"], "includes", "REHAB_THERAPY_ROUTINE")
        ],
    ),

    make_rule(
        "R_FALL_OR_NEAR_FALL_LOGGED",
        "Fall or near-fall has been logged recently.",
        0.80,
        ["FALL_OR_NEAR_FALL_CONTEXT"],
        ["observation_counts_7d.FALL_OR_NEAR_FALL"],
        lambda ctx: obs_count(ctx, "FALL_OR_NEAR_FALL") >= 1,
        lambda ctx: [
            evidence_ref("observation_counts_7d.FALL_OR_NEAR_FALL", obs_count(ctx, "FALL_OR_NEAR_FALL"), ">=", 1)
        ],
    ),

    make_rule(
        "R_CAREGIVER_PROVIDER_REVIEW_REQUESTED",
        "Caregiver requested provider review recently.",
        0.75,
        ["CAREGIVER_PROVIDER_REVIEW_REQUEST"],
        ["observation_counts_7d.CAREGIVER_WANTS_PROVIDER_REVIEW"],
        lambda ctx: obs_count(ctx, "CAREGIVER_WANTS_PROVIDER_REVIEW") >= 1,
        lambda ctx: [
            evidence_ref("observation_counts_7d.CAREGIVER_WANTS_PROVIDER_REVIEW", obs_count(ctx, "CAREGIVER_WANTS_PROVIDER_REVIEW"), ">=", 1)
        ],
    ),

    make_rule(
        "R_RECENT_SEVERITY2_OR_PROVIDER_RELEVANT_EVENT",
        "Recent structured event may be useful for provider review.",
        0.45,
        ["CAREGIVER_PROVIDER_REVIEW_REQUEST"],
        ["uc2_recent_events"],
        lambda ctx: len(ctx.get("uc2_recent_events", [])) >= 1 or obs_count(ctx, "FALL_OR_NEAR_FALL") >= 1 or obs_count(ctx, "SEIZURE_LIKE_EVENT_REPORTED") >= 1,
        lambda ctx: [
            evidence_ref("uc2_recent_events_count", len(ctx.get("uc2_recent_events", [])), ">=", 1),
            evidence_ref("observation_counts_7d.FALL_OR_NEAR_FALL", obs_count(ctx, "FALL_OR_NEAR_FALL")),
            evidence_ref("observation_counts_7d.SEIZURE_LIKE_EVENT_REPORTED", obs_count(ctx, "SEIZURE_LIKE_EVENT_REPORTED")),
        ],
    ),
]

In [34]:
def strip_callable_rule(rule: Dict[str, Any]) -> Dict[str, Any]:
    return {
        k: v for k, v in rule.items()
        if k not in ["evaluate", "build_evidence"]
    }

RULE_REGISTRY_METADATA = [strip_callable_rule(r) for r in RULE_REGISTRY]
write_json(OUTPUT_DIR / "uc4_rule_registry.json", RULE_REGISTRY_METADATA)

pd.DataFrame(RULE_REGISTRY_METADATA).head()

,rule_code,rule_registry_version,description,weight,applies_to_templates,evidence_fields,safety_tags,enabled
0,R_FATIGUE_RECURRENCE,uc4_rule_registry_v0.1.0,Unusual fatigue has been logged at least twice...,0.40,[MEDICATION_WINDOW_FATIGUE_TRACKING],[observation_counts_7d.UNUSUAL_FATIGUE],"{'diagnostic': False, 'treatment_recommendatio...",True
1,R_MED_WATCH_AREA_MATCH_FATIGUE,uc4_rule_registry_v0.1.0,Medication profile includes sleepiness/fatigue...,0.30,[MEDICATION_WINDOW_FATIGUE_TRACKING],[medication_watch_areas],"{'diagnostic': False, 'treatment_recommendatio...",True
2,R_MED_TIMING_CONTEXT_MISSING,uc4_rule_registry_v0.1.0,Medication timing context has been missing in ...,0.20,"[MEDICATION_WINDOW_FATIGUE_TRACKING, MISSED_DE...",[missing_schema_field_counts.timing_relative_t...,"{'diagnostic': False, 'treatment_recommendatio...",True
3,R_MISSED_DELAYED_MEDICATION_LOGGED,uc4_rule_registry_v0.1.0,Missed or delayed medication was logged recently.,0.50,[MISSED_DELAYED_MEDICATION_CONTEXT],[observation_counts_7d.MISSED_OR_DELAYED_MEDIC...,"{'diagnostic': False, 'treatment_recommendatio...",True
4,R_TRANSFER_DISCOMFORT_RECURRENCE,uc4_rule_registry_v0.1.0,Discomfort and transfer/positioning context ha...,0.55,[TRANSFER_DISCOMFORT_TRACKING],"[observation_counts_7d.PAIN_OR_DISCOMFORT, obs...","{'diagnostic': False, 'treatment_recommendatio...",True


In [36]:
def validate_rule_registry(rule_registry: List[Dict[str, Any]], template_registry: Dict[str, Any]) -> Dict[str, Any]:
    errors = []
    warnings = []

    required_keys = [
        "rule_code",
        "description",
        "weight",
        "applies_to_templates",
        "evidence_fields",
        "evaluate",
        "build_evidence",
        "safety_tags",
        "enabled",
    ]

    seen_codes = set()

    for rule in rule_registry:
        code = rule.get("rule_code")

        for key in required_keys:
            if key not in rule:
                errors.append(f"Rule {code} missing required key: {key}")

        if code in seen_codes:
            errors.append(f"Duplicate rule_code: {code}")
        seen_codes.add(code)

        if not isinstance(rule.get("weight"), (int, float)) or not (0 <= rule["weight"] <= 1):
            errors.append(f"Rule {code} has invalid weight: {rule.get('weight')}")

        for template_id in rule.get("applies_to_templates", []):
            if template_id not in template_registry:
                errors.append(f"Rule {code} references unknown template_id: {template_id}")

        safety = rule.get("safety_tags", {})
        for flag in [
            "diagnostic",
            "treatment_recommendation",
            "medication_causality",
            "emergency_override",
            "slm_scoring",
            "free_text_scoring",
        ]:
            if safety.get(flag) is not False:
                errors.append(f"Rule {code} safety flag {flag} must be False")

    return {
        "valid": len(errors) == 0,
        "errors": errors,
        "warnings": warnings,
        "rule_count": len(rule_registry),
        "template_count": len(template_registry),
    }

rule_registry_validation = validate_rule_registry(RULE_REGISTRY, TEMPLATE_REGISTRY)
write_json(OUTPUT_DIR / "uc4_rule_registry_validation.json", rule_registry_validation)
rule_registry_validation

{'valid': True,
 'errors': [],
 'warnings': [],
 'rule_count': 23,
 'template_count': 11}

In [38]:
def fire_rules_for_template(template_id: str, ctx: Dict[str, Any], rule_registry: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    fired = []

    for rule in rule_registry:
        if not rule.get("enabled", True):
            continue

        if template_id not in rule["applies_to_templates"]:
            continue

        result = bool(rule["evaluate"](ctx))

        if result:
            fired.append({
                "patient_id": ctx["patient_id"],
                "template_id": template_id,
                "rule_code": rule["rule_code"],
                "rule_registry_version": RULE_REGISTRY_VERSION,
                "weight": rule["weight"],
                "description": rule["description"],
                "evidence_refs": rule["build_evidence"](ctx),
                "safety_tags": rule["safety_tags"],
            })

    return fired

def fire_rules_for_patient(ctx: Dict[str, Any]) -> Dict[str, List[Dict[str, Any]]]:
    result = {}
    for template_id in ctx["activated_templates"]:
        if template_id not in TEMPLATE_REGISTRY:
            continue
        result[template_id] = fire_rules_for_template(template_id, ctx, RULE_REGISTRY)
    return result

In [40]:
def is_blind_spot(template_id: str, ctx: Dict[str, Any]) -> bool:
    schema = TEMPLATE_REGISTRY[template_id]["what_to_log_next_schema"]
    field_ids = [f["field_id"] for f in schema]
    return any(missing_field_count(ctx, field_id) >= 2 for field_id in field_ids)

def calculate_priority_score(
    fired_rules: List[Dict[str, Any]],
    template_id: str,
    ctx: Dict[str, Any],
) -> Dict[str, Any]:
    rule_score = min(1.0, sum(float(r["weight"]) for r in fired_rules))

    blind_spot = is_blind_spot(template_id, ctx)
    is_previous_priority = template_id in ctx.get("previous_priority_template_ids", [])
    caregiver_confirmed_useful = template_id in ctx.get("caregiver_preference_signals", {}).get("confirmed_useful_template_ids", [])
    caregiver_dismissed_recently = template_id in ctx.get("caregiver_preference_signals", {}).get("dismissed_template_ids", [])

    blind_spot_bonus = 0.10 if blind_spot else 0.0
    usefulness_bonus = 0.10 if caregiver_confirmed_useful else 0.0
    repeat_penalty = 0.15 if is_previous_priority else 0.0
    dismiss_penalty = 0.25 if caregiver_dismissed_recently else 0.0

    raw_score = (
        rule_score * 0.65
        + blind_spot_bonus
        + usefulness_bonus
        - repeat_penalty
        - dismiss_penalty
    )

    final_score = clamp01(raw_score)

    return {
        "scoring_version": SCORING_VERSION,
        "template_id": template_id,
        "rule_score": round(rule_score, 4),
        "blind_spot": blind_spot,
        "blind_spot_bonus": blind_spot_bonus,
        "is_previous_priority": is_previous_priority,
        "repeat_penalty": repeat_penalty,
        "caregiver_confirmed_useful": caregiver_confirmed_useful,
        "usefulness_bonus": usefulness_bonus,
        "caregiver_dismissed_recently": caregiver_dismissed_recently,
        "dismiss_penalty": dismiss_penalty,
        "raw_score": round(raw_score, 4),
        "final_score": round(final_score, 4),
        "fired_rule_codes": [r["rule_code"] for r in fired_rules],
    }

In [42]:
def generate_candidates_for_patient(ctx: Dict[str, Any]) -> List[Dict[str, Any]]:
    # UC1 active emergency pause: UC4 must not compete with emergency fast path.
    if ctx["safety_flags"].get("uc1_emergency_active") is True:
        return []

    candidates = []
    fired_by_template = fire_rules_for_patient(ctx)

    for template_id, fired_rules in fired_by_template.items():
        if not fired_rules:
            continue

        scoring_trace = calculate_priority_score(fired_rules, template_id, ctx)

        # Require non-trivial score.
        if scoring_trace["final_score"] <= 0:
            continue

        candidates.append({
            "patient_id": ctx["patient_id"],
            "patient_first_name": ctx["patient_first_name"],
            "template_id": template_id,
            "template_registry_version": TEMPLATE_REGISTRY_VERSION,
            "rule_registry_version": RULE_REGISTRY_VERSION,
            "scoring_version": SCORING_VERSION,
            "priority_type": TEMPLATE_REGISTRY[template_id]["priority_type"],
            "final_score": scoring_trace["final_score"],
            "fired_rules": fired_rules,
            "scoring_trace": scoring_trace,
            "route": "UC4_ROUTINE_CHECKLIST",
            "free_text_used_for_scoring": False,
            "slm_used_for_scoring": False,
            "synthetic": True,
        })

    candidates = sorted(candidates, key=lambda x: x["final_score"], reverse=True)
    return candidates

CANDIDATES_BY_PATIENT = {
    pid: generate_candidates_for_patient(ctx)
    for pid, ctx in RULE_CONTEXTS_BY_PATIENT.items()
}

write_json(OUTPUT_DIR / "uc4_candidates_by_patient.json", CANDIDATES_BY_PATIENT)

flat_candidates = []
for pid, candidates in CANDIDATES_BY_PATIENT.items():
    for c in candidates:
        flat_candidates.append({
            "patient_id": pid,
            "template_id": c["template_id"],
            "priority_type": c["priority_type"],
            "final_score": c["final_score"],
            "fired_rule_codes": "|".join(c["scoring_trace"]["fired_rule_codes"]),
            "rule_score": c["scoring_trace"]["rule_score"],
            "blind_spot": c["scoring_trace"]["blind_spot"],
            "is_previous_priority": c["scoring_trace"]["is_previous_priority"],
        })

candidates_df = pd.DataFrame(flat_candidates)
candidates_df.to_csv(OUTPUT_DIR / "uc4_micro_priority_candidates_rule_registry.csv", index=False)
candidates_df

,patient_id,template_id,priority_type,final_score,fired_rule_codes,rule_score,blind_spot,is_previous_priority
0,Mike_DEMO_001,TRANSFER_DISCOMFORT_TRACKING,recurring_concern,0.7000,R_TRANSFER_DISCOMFORT_RECURRENCE|R_TRANSFER_CO...,1.00,True,True
1,Mike_DEMO_001,MEDICATION_WINDOW_FATIGUE_TRACKING,recurring_concern,0.6850,R_FATIGUE_RECURRENCE|R_MED_WATCH_AREA_MATCH_FA...,0.90,True,False
2,Mike_DEMO_001,SKIN_PRESSURE_AFTER_SEATED_PERIOD,blind_spot,0.6500,R_LOW_MOVEMENT_INCREASE|R_SKIN_PRESSURE_FOCUS|...,1.00,False,False
3,Mike_DEMO_001,CAREGIVER_REPORTED_SEIZURE_LIKE_EVENT_CONTEXT,provider_review_support,0.6500,R_UNUSUAL_RESPONSIVENESS_LOGGED|R_CAREGIVER_RE...,1.00,False,False
4,Mike_DEMO_001,BOWEL_ROUTINE_DISCOMFORT_CONTEXT,recurring_concern,0.4875,R_BOWEL_BLADDER_FOCUS|R_BOWEL_ROUTINE_CHANGE_L...,0.75,False,False
5,Mike_DEMO_001,CAREGIVER_PROVIDER_REVIEW_REQUEST,provider_review_support,0.2925,R_RECENT_SEVERITY2_OR_PROVIDER_RELEVANT_EVENT,0.45,False,False
6,Mike_DEMO_001,MISSED_DELAYED_MEDICATION_CONTEXT,emerging_pattern,0.2300,R_MED_TIMING_CONTEXT_MISSING,0.20,True,False
7,JAMES_DEMO_001,FALL_OR_NEAR_FALL_CONTEXT,provider_review_support,0.6500,R_TRANSFER_CONTEXT_CLUSTER|R_FALL_OR_NEAR_FALL...,1.00,False,False
8,JAMES_DEMO_001,CAREGIVER_PROVIDER_REVIEW_REQUEST,provider_review_support,0.6500,R_CAREGIVER_PROVIDER_REVIEW_REQUESTED|R_RECENT...,1.00,False,False
9,JAMES_DEMO_001,THERAPY_REHAB_ROUTINE_DIFFICULTY,recurring_concern,0.5850,R_THERAPY_ROUTINE_DIFFICULTY|R_REHAB_FOCUS,0.90,False,False


In [44]:
def select_top_priorities(candidates: List[Dict[str, Any]], max_cards: int = 3) -> List[Dict[str, Any]]:
    selected = []
    used_priority_types = set()

    # First pass: diversify priority types.
    for cand in candidates:
        if len(selected) >= max_cards:
            break
        ptype = cand["priority_type"]
        if ptype not in used_priority_types:
            selected.append(cand)
            used_priority_types.add(ptype)

    # Second pass: fill if fewer than max.
    for cand in candidates:
        if len(selected) >= max_cards:
            break
        if cand not in selected:
            selected.append(cand)

    return selected

TOP_PRIORITIES_BY_PATIENT = {
    pid: select_top_priorities(candidates, max_cards=3)
    for pid, candidates in CANDIDATES_BY_PATIENT.items()
}

flat_top = []
for pid, priorities in TOP_PRIORITIES_BY_PATIENT.items():
    for rank, p in enumerate(priorities, start=1):
        flat_top.append({
            "patient_id": pid,
            "rank": rank,
            "template_id": p["template_id"],
            "priority_type": p["priority_type"],
            "final_score": p["final_score"],
            "fired_rule_codes": "|".join(p["scoring_trace"]["fired_rule_codes"]),
        })

top_df = pd.DataFrame(flat_top)
top_df.to_csv(OUTPUT_DIR / "uc4_top_structured_priorities_rule_registry.csv", index=False)
top_df

,patient_id,rank,template_id,priority_type,final_score,fired_rule_codes
0,Mike_DEMO_001,1,TRANSFER_DISCOMFORT_TRACKING,recurring_concern,0.7000,R_TRANSFER_DISCOMFORT_RECURRENCE|R_TRANSFER_CO...
1,Mike_DEMO_001,2,SKIN_PRESSURE_AFTER_SEATED_PERIOD,blind_spot,0.6500,R_LOW_MOVEMENT_INCREASE|R_SKIN_PRESSURE_FOCUS|...
2,Mike_DEMO_001,3,CAREGIVER_REPORTED_SEIZURE_LIKE_EVENT_CONTEXT,provider_review_support,0.6500,R_UNUSUAL_RESPONSIVENESS_LOGGED|R_CAREGIVER_RE...
3,JAMES_DEMO_001,1,FALL_OR_NEAR_FALL_CONTEXT,provider_review_support,0.6500,R_TRANSFER_CONTEXT_CLUSTER|R_FALL_OR_NEAR_FALL...
4,JAMES_DEMO_001,2,THERAPY_REHAB_ROUTINE_DIFFICULTY,recurring_concern,0.5850,R_THERAPY_ROUTINE_DIFFICULTY|R_REHAB_FOCUS
5,JAMES_DEMO_001,3,CAREGIVER_PROVIDER_REVIEW_REQUEST,provider_review_support,0.6500,R_CAREGIVER_PROVIDER_REVIEW_REQUESTED|R_RECENT...
6,SOFIA_DEMO_001,1,BOWEL_ROUTINE_DISCOMFORT_CONTEXT,recurring_concern,0.6000,R_BOWEL_BLADDER_FOCUS|R_BOWEL_ROUTINE_CHANGE_L...
7,SOFIA_DEMO_001,2,MISSED_DELAYED_MEDICATION_CONTEXT,emerging_pattern,0.5550,R_MED_TIMING_CONTEXT_MISSING|R_MISSED_DELAYED_...
8,SOFIA_DEMO_001,3,CAREGIVER_PROVIDER_REVIEW_REQUEST,provider_review_support,0.2925,R_RECENT_SEVERITY2_OR_PROVIDER_RELEVANT_EVENT
9,ELENA_DEMO_001,1,BREATHING_CONCERN_CONTEXT,emerging_pattern,0.7500,R_BREATHING_FOCUS|R_BREATHING_CONCERN_LOGGED|R...


In [46]:
UNSAFE_LANGUAGE_PATTERNS = [
    r"\bdiagnos",
    r"\bcaused by medication\b",
    r"\bmedication caused\b",
    r"\bchange (the )?dose\b",
    r"\bstop (the )?medication\b",
    r"\bstart (a )?medication\b",
    r"\bseizure detected\b",
    r"\bwound detected\b",
    r"\binfection detected\b",
    r"\bthis is a seizure\b",
    r"\bthis is an emergency\b",  # UC4 should defer to emergency plan, not independently declare.
]

def contains_unsafe_language(text: str) -> List[str]:
    hits = []
    lower = text.lower()
    for pattern in UNSAFE_LANGUAGE_PATTERNS:
        if re.search(pattern, lower):
            hits.append(pattern)
    return hits

def summarize_evidence(fired_rules: List[Dict[str, Any]]) -> str:
    parts = []
    for r in fired_rules:
        for ev in r.get("evidence_refs", []):
            field = ev.get("field_path")
            value = ev.get("value")
            comparator = ev.get("comparator", "")
            threshold = ev.get("threshold", "")
            if comparator:
                parts.append(f"{field}={value} ({comparator} {threshold})")
            else:
                parts.append(f"{field}={value}")
    return "; ".join(parts)

def render_priority_card(candidate: Dict[str, Any]) -> Dict[str, Any]:
    template_id = candidate["template_id"]
    template = TEMPLATE_REGISTRY[template_id]

    patient_first_name = candidate["patient_first_name"]
    fired_rule_codes = candidate["scoring_trace"]["fired_rule_codes"]
    evidence_summary = summarize_evidence(candidate["fired_rules"])

    caregiver_summary = template["caregiver_summary_template"].format(
        patient_first_name=patient_first_name,
        fired_rule_codes=", ".join(fired_rule_codes),
        evidence_summary=evidence_summary,
    )

    provider_summary = template["provider_summary_template"].format(
        patient_first_name=patient_first_name,
        fired_rule_codes=", ".join(fired_rule_codes),
        evidence_summary=evidence_summary,
    )

    card = {
        "patient_id": candidate["patient_id"],
        "synthetic": True,
        "engine_version": ENGINE_VERSION,
        "schema_version": SCHEMA_VERSION,
        "template_registry_version": TEMPLATE_REGISTRY_VERSION,
        "rule_registry_version": RULE_REGISTRY_VERSION,
        "scoring_version": SCORING_VERSION,

        "template_id": template_id,
        "template_version": template["template_version"],
        "title": template["title"],
        "priority_type": template["priority_type"],
        "score": candidate["final_score"],
        "route": candidate["route"],

        "caregiver_summary": caregiver_summary,
        "what_to_log_next_schema": template["what_to_log_next_schema"],

        "provider_summary_draft": provider_summary,

        "fired_rules": candidate["fired_rules"],
        "scoring_trace": candidate["scoring_trace"],

        "safety": {
            "diagnosis": False,
            "treatment_recommendation": False,
            "medication_causality": False,
            "emergency_override": False,
            "slm_used_for_scoring": False,
            "free_text_used_for_scoring": False,
            "caregiver_visible_text_from_template": True,
        },

        "created_at": RUN_TIMESTAMP,
    }

    return card

In [48]:
STRUCTURED_PRIORITY_CARDS_BY_PATIENT = {
    pid: [render_priority_card(p) for p in priorities]
    for pid, priorities in TOP_PRIORITIES_BY_PATIENT.items()
}

write_json(OUTPUT_DIR / "uc4_structured_priority_cards_by_patient.json", STRUCTURED_PRIORITY_CARDS_BY_PATIENT)

preview_rows = []
for pid, cards in STRUCTURED_PRIORITY_CARDS_BY_PATIENT.items():
    for i, card in enumerate(cards, start=1):
        preview_rows.append({
            "patient_id": pid,
            "rank": i,
            "template_id": card["template_id"],
            "title": card["title"],
            "score": card["score"],
            "caregiver_summary": card["caregiver_summary"],
            "schema_fields": "|".join([f["field_id"] for f in card["what_to_log_next_schema"]]),
            "fired_rules": "|".join([r["rule_code"] for r in card["fired_rules"]]),
        })

preview_df = pd.DataFrame(preview_rows)
preview_df.to_csv(OUTPUT_DIR / "uc4_priority_card_preview.csv", index=False)
preview_df

,patient_id,rank,template_id,title,score,caregiver_summary,schema_fields,fired_rules
0,Mike_DEMO_001,1,TRANSFER_DISCOMFORT_TRACKING,Discomfort during transfers or positioning,0.7000,Recent observations suggest it may be useful t...,transfer_phase|position_before|position_after|...,R_TRANSFER_DISCOMFORT_RECURRENCE|R_TRANSFER_CO...
1,Mike_DEMO_001,2,SKIN_PRESSURE_AFTER_SEATED_PERIOD,Skin and pressure check after seated time,0.6500,Mike's care profile includes skin and pressure...,position_before_check|skin_area_checked|skin_o...,R_LOW_MOVEMENT_INCREASE|R_SKIN_PRESSURE_FOCUS|...
2,Mike_DEMO_001,3,CAREGIVER_REPORTED_SEIZURE_LIKE_EVENT_CONTEXT,Caregiver-reported seizure-like event context,0.6500,A caregiver-reported seizure-like event can be...,event_reported_by_caregiver|responsiveness_cha...,R_UNUSUAL_RESPONSIVENESS_LOGGED|R_CAREGIVER_RE...
3,JAMES_DEMO_001,1,FALL_OR_NEAR_FALL_CONTEXT,Fall or near-fall context,0.6500,A fall or near-fall can be important context f...,fall_or_near_fall_seen|position_before|caregiv...,R_TRANSFER_CONTEXT_CLUSTER|R_FALL_OR_NEAR_FALL...
4,JAMES_DEMO_001,2,THERAPY_REHAB_ROUTINE_DIFFICULTY,Therapy or rehab routine difficulty,0.5850,James's care plan includes therapy or rehab ro...,therapy_difficulty_seen|therapy_context|discom...,R_THERAPY_ROUTINE_DIFFICULTY|R_REHAB_FOCUS
5,JAMES_DEMO_001,3,CAREGIVER_PROVIDER_REVIEW_REQUEST,Prepare a provider review summary,0.6500,Recent structured logs suggest it may be usefu...,provider_review_reason|caregiver_wants_provide...,R_CAREGIVER_PROVIDER_REVIEW_REQUESTED|R_RECENT...
6,SOFIA_DEMO_001,1,BOWEL_ROUTINE_DISCOMFORT_CONTEXT,Bowel/bladder routine and discomfort context,0.6000,Sofia's care profile includes bowel or bladder...,bowel_bladder_change_seen|hydration_or_appetit...,R_BOWEL_BLADDER_FOCUS|R_BOWEL_ROUTINE_CHANGE_L...
7,SOFIA_DEMO_001,2,MISSED_DELAYED_MEDICATION_CONTEXT,Missed or delayed medication context,0.5550,Recent logs suggest medication timing may be w...,dose_missed_or_delayed|timing_relative_to_medi...,R_MED_TIMING_CONTEXT_MISSING|R_MISSED_DELAYED_...
8,SOFIA_DEMO_001,3,CAREGIVER_PROVIDER_REVIEW_REQUEST,Prepare a provider review summary,0.2925,Recent structured logs suggest it may be usefu...,provider_review_reason|caregiver_wants_provide...,R_RECENT_SEVERITY2_OR_PROVIDER_RELEVANT_EVENT
9,ELENA_DEMO_001,1,BREATHING_CONCERN_CONTEXT,Breathing concern context,0.7500,Elena's care profile includes breathing as an ...,breathing_concern_present|breathing_context|ca...,R_BREATHING_FOCUS|R_BREATHING_CONCERN_LOGGED|R...


In [50]:
def example_response_for_card(card: Dict[str, Any]) -> Dict[str, Any]:
    responses = {}
    schema = card["what_to_log_next_schema"]

    for field in schema:
        fid = field["field_id"]
        input_type = field["input_type"]

        if input_type == "select":
            # Select first non-NOT_SURE option when possible.
            options = field.get("options", [])
            chosen = None
            for opt in options:
                if opt["value"] not in ["NOT_SURE", "NONE_OBSERVED"]:
                    chosen = opt["value"]
                    break
            responses[fid] = chosen or options[0]["value"]

        elif input_type == "multiselect":
            options = field.get("options", [])
            chosen = []
            for opt in options:
                if opt["value"] not in ["NOT_SURE", "NONE_OBSERVED"]:
                    chosen.append(opt["value"])
                    break
            responses[fid] = chosen or [options[0]["value"]]

        elif input_type == "boolean":
            responses[fid] = True

        elif input_type == "number":
            responses[fid] = None

        else:
            responses[fid] = None

    return {
        "response_id": f"response_{card['patient_id']}_{card['template_id']}",
        "source_priority_template_id": card["template_id"],
        "patient_id": card["patient_id"],
        "schema_version": SCHEMA_VERSION,
        "template_registry_version": TEMPLATE_REGISTRY_VERSION,
        "created_at": now_iso(),
        "responses": responses,
        "observation_codes": [],  # Populated by mapping layer below if needed.
        "context_codes": [],
        "free_text": None,
        "free_text_used_for_scoring": False,
        "slm_used_for_scoring": False,
        "audit": {
            "all_response_fields_from_schema": True,
            "no_free_text_scoring": True,
            "no_slm_scoring": True,
        }
    }

CAREGIVER_RESPONSE_EXAMPLES = []
for pid, cards in STRUCTURED_PRIORITY_CARDS_BY_PATIENT.items():
    for card in cards:
        CAREGIVER_RESPONSE_EXAMPLES.append(example_response_for_card(card))

write_json(OUTPUT_DIR / "uc4_caregiver_response_examples.json", CAREGIVER_RESPONSE_EXAMPLES)
CAREGIVER_RESPONSE_EXAMPLES[:2]

[{'response_id': 'response_Mike_DEMO_001_TRANSFER_DISCOMFORT_TRACKING',
  'source_priority_template_id': 'TRANSFER_DISCOMFORT_TRACKING',
  'patient_id': 'Mike_DEMO_001',
  'schema_version': 'uc4_schema_v0.1.0',
  'template_registry_version': 'uc4_template_registry_v0.1.0',
  'created_at': '2026-07-16T19:21:35.512442+00:00',
  'responses': {'transfer_phase': 'BEFORE_TRANSFER',
   'position_before': 'BED',
   'position_after': 'BED',
   'discomfort_cues': ['FACIAL_GRIMACE'],
   'repositioning_helped': 'YES'},
  'observation_codes': [],
  'context_codes': [],
  'free_text': None,
  'free_text_used_for_scoring': False,
  'slm_used_for_scoring': False,
  'audit': {'all_response_fields_from_schema': True,
   'no_free_text_scoring': True,
   'no_slm_scoring': True}},
 {'response_id': 'response_Mike_DEMO_001_SKIN_PRESSURE_AFTER_SEATED_PERIOD',
  'source_priority_template_id': 'SKIN_PRESSURE_AFTER_SEATED_PERIOD',
  'patient_id': 'Mike_DEMO_001',
  'schema_version': 'uc4_schema_v0.1.0',
  'templ

In [52]:
def map_response_to_next_cycle_event(response: Dict[str, Any]) -> Dict[str, Any]:
    template_id = response["source_priority_template_id"]
    r = response["responses"]

    observation_codes = []
    context_codes = []

    if template_id == "MEDICATION_WINDOW_FATIGUE_TRACKING":
        if r.get("fatigue_present") == "YES":
            observation_codes.append("UNUSUAL_FATIGUE")
        if r.get("timing_relative_to_medication") in ["BEFORE_MEDICATION", "WITHIN_2_HOURS_AFTER", "LATER_IN_DAY"]:
            context_codes.append("AROUND_MEDICATION_TIME")

    elif template_id == "TRANSFER_DISCOMFORT_TRACKING":
        observation_codes.extend(["TRANSFER_OR_POSITIONING_CONTEXT"])
        if "discomfort_cues" in r and r["discomfort_cues"]:
            if "NONE_OBSERVED" not in r["discomfort_cues"]:
                observation_codes.append("PAIN_OR_DISCOMFORT")
        if r.get("transfer_phase") in ["BEFORE_TRANSFER", "DURING_TRANSFER", "AFTER_TRANSFER"]:
            context_codes.append("DURING_TRANSFER")

    elif template_id == "SKIN_PRESSURE_AFTER_SEATED_PERIOD":
        if r.get("skin_or_pressure_concern_seen") == "YES":
            observation_codes.append("SKIN_OR_PRESSURE_CONCERN")
        if r.get("position_before_check") == "CHAIR_OR_WHEELCHAIR":
            context_codes.append("WHILE_SITTING_OR_POSITIONED")

    elif template_id == "BOWEL_ROUTINE_DISCOMFORT_CONTEXT":
        if r.get("bowel_bladder_change_seen") == "YES":
            observation_codes.append("BOWEL_OR_BLADDER_CHANGE")
        if r.get("hydration_or_appetite_change") in ["LESS_THAN_USUAL", "MUCH_LESS_THAN_USUAL"]:
            observation_codes.append("APPETITE_OR_HYDRATION_CHANGE")
        context_codes.append("BATHROOM_OR_BOWEL_BLADDER")

    elif template_id == "BREATHING_CONCERN_CONTEXT":
        if r.get("breathing_concern_present") == "YES":
            observation_codes.append("BREATHING_CONCERN")
        if r.get("breathing_context") == "AROUND_MEDICATION_TIME":
            context_codes.append("AROUND_MEDICATION_TIME")
        elif r.get("breathing_context") == "AFTER_ACTIVITY":
            context_codes.append("AFTER_ACTIVITY_OR_THERAPY")
        elif r.get("breathing_context") == "DURING_SLEEP_OR_NIGHT":
            context_codes.append("DURING_SLEEP_OR_NIGHT")

    elif template_id == "UNUSUAL_RESPONSIVENESS_CONTEXT":
        if r.get("responsiveness_change") not in [None, "USUAL", "NOT_SURE"]:
            observation_codes.append("UNUSUAL_RESPONSIVENESS")

    elif template_id == "CAREGIVER_REPORTED_SEIZURE_LIKE_EVENT_CONTEXT":
        if r.get("event_reported_by_caregiver") == "YES":
            observation_codes.append("SEIZURE_LIKE_EVENT_REPORTED")
        if r.get("responsiveness_change") not in [None, "USUAL", "NOT_SURE"]:
            observation_codes.append("UNUSUAL_RESPONSIVENESS")

    elif template_id == "THERAPY_REHAB_ROUTINE_DIFFICULTY":
        if r.get("therapy_difficulty_seen") == "YES":
            observation_codes.append("THERAPY_ROUTINE_DIFFICULTY")
        context_codes.append("AFTER_ACTIVITY_OR_THERAPY")

    elif template_id == "FALL_OR_NEAR_FALL_CONTEXT":
        if r.get("fall_or_near_fall_seen") == "YES":
            observation_codes.append("FALL_OR_NEAR_FALL")

    elif template_id == "CAREGIVER_PROVIDER_REVIEW_REQUEST":
        if r.get("caregiver_wants_provider_review") == "YES":
            observation_codes.append("CAREGIVER_WANTS_PROVIDER_REVIEW")

    return {
        "patient_id": response["patient_id"],
        "source": "UC4_CAREGIVER_RESPONSE",
        "event_type": "STRUCTURED_OBSERVATION",
        "source_priority_template_id": template_id,
        "days_ago": 0,
        "observation_codes": sorted(set(observation_codes)),
        "context_codes": sorted(set(context_codes)),
        "severity": 1,
        "free_text_used_for_scoring": False,
        "slm_used_for_scoring": False,
        "created_at": response["created_at"],
    }

NEXT_CYCLE_RESPONSE_EVENTS = [
    map_response_to_next_cycle_event(resp)
    for resp in CAREGIVER_RESPONSE_EXAMPLES
]

write_json(OUTPUT_DIR / "uc4_next_cycle_response_events.json", NEXT_CYCLE_RESPONSE_EVENTS)
pd.DataFrame(NEXT_CYCLE_RESPONSE_EVENTS).to_csv(OUTPUT_DIR / "uc4_next_cycle_response_events.csv", index=False)
pd.DataFrame(NEXT_CYCLE_RESPONSE_EVENTS).head()

,patient_id,source,event_type,source_priority_template_id,days_ago,observation_codes,context_codes,severity,free_text_used_for_scoring,slm_used_for_scoring,created_at
0,Mike_DEMO_001,UC4_CAREGIVER_RESPONSE,STRUCTURED_OBSERVATION,TRANSFER_DISCOMFORT_TRACKING,0,"[PAIN_OR_DISCOMFORT, TRANSFER_OR_POSITIONING_C...",[DURING_TRANSFER],1,False,False,2026-07-16T19:21:35.512442+00:00
1,Mike_DEMO_001,UC4_CAREGIVER_RESPONSE,STRUCTURED_OBSERVATION,SKIN_PRESSURE_AFTER_SEATED_PERIOD,0,[SKIN_OR_PRESSURE_CONCERN],[],1,False,False,2026-07-16T19:21:35.512442+00:00
2,Mike_DEMO_001,UC4_CAREGIVER_RESPONSE,STRUCTURED_OBSERVATION,CAREGIVER_REPORTED_SEIZURE_LIKE_EVENT_CONTEXT,0,[SEIZURE_LIKE_EVENT_REPORTED],[],1,False,False,2026-07-16T19:21:35.512442+00:00
3,JAMES_DEMO_001,UC4_CAREGIVER_RESPONSE,STRUCTURED_OBSERVATION,FALL_OR_NEAR_FALL_CONTEXT,0,[FALL_OR_NEAR_FALL],[],1,False,False,2026-07-16T19:21:35.512442+00:00
4,JAMES_DEMO_001,UC4_CAREGIVER_RESPONSE,STRUCTURED_OBSERVATION,THERAPY_REHAB_ROUTINE_DIFFICULTY,0,[THERAPY_ROUTINE_DIFFICULTY],[AFTER_ACTIVITY_OR_THERAPY],1,False,False,2026-07-16T19:21:35.512442+00:00


In [54]:
def derive_next_cycle_features(response_events: List[Dict[str, Any]]) -> Dict[str, Any]:
    features_by_patient = {}

    for pid in SYNTHETIC_PATIENT_PROFILES.keys():
        evs = [e for e in response_events if e["patient_id"] == pid]

        features_by_patient[pid] = {
            "patient_id": pid,
            "derived_from": "UC4_CAREGIVER_RESPONSE",
            "free_text_used_for_scoring": False,
            "slm_used_for_scoring": False,
            "response_event_count": len(evs),
            "transfer_discomfort_logs_7d": sum(
                "TRANSFER_OR_POSITIONING_CONTEXT" in e["observation_codes"] and "PAIN_OR_DISCOMFORT" in e["observation_codes"]
                for e in evs
            ),
            "skin_pressure_concerns_reported": sum(
                "SKIN_OR_PRESSURE_CONCERN" in e["observation_codes"]
                for e in evs
            ),
            "medication_timing_context_completion_count": sum(
                "AROUND_MEDICATION_TIME" in e["context_codes"]
                for e in evs
            ),
            "provider_review_requests": sum(
                "CAREGIVER_WANTS_PROVIDER_REVIEW" in e["observation_codes"]
                for e in evs
            ),
            "breathing_concern_logs": sum(
                "BREATHING_CONCERN" in e["observation_codes"]
                for e in evs
            ),
            "unusual_responsiveness_logs": sum(
                "UNUSUAL_RESPONSIVENESS" in e["observation_codes"]
                for e in evs
            ),
        }

    return features_by_patient

NEXT_CYCLE_DERIVED_FEATURES = derive_next_cycle_features(NEXT_CYCLE_RESPONSE_EVENTS)
write_json(OUTPUT_DIR / "uc4_next_cycle_derived_features_from_caregiver_responses.json", NEXT_CYCLE_DERIVED_FEATURES)
NEXT_CYCLE_DERIVED_FEATURES

{'Mike_DEMO_001': {'patient_id': 'Mike_DEMO_001',
  'derived_from': 'UC4_CAREGIVER_RESPONSE',
  'free_text_used_for_scoring': False,
  'slm_used_for_scoring': False,
  'response_event_count': 3,
  'transfer_discomfort_logs_7d': 1,
  'skin_pressure_concerns_reported': 1,
  'medication_timing_context_completion_count': 0,
  'provider_review_requests': 0,
  'breathing_concern_logs': 0,
  'unusual_responsiveness_logs': 0},
 'JAMES_DEMO_001': {'patient_id': 'JAMES_DEMO_001',
  'derived_from': 'UC4_CAREGIVER_RESPONSE',
  'free_text_used_for_scoring': False,
  'slm_used_for_scoring': False,
  'response_event_count': 3,
  'transfer_discomfort_logs_7d': 0,
  'skin_pressure_concerns_reported': 0,
  'medication_timing_context_completion_count': 0,
  'provider_review_requests': 1,
  'breathing_concern_logs': 0,
  'unusual_responsiveness_logs': 0},
 'SOFIA_DEMO_001': {'patient_id': 'SOFIA_DEMO_001',
  'derived_from': 'UC4_CAREGIVER_RESPONSE',
  'free_text_used_for_scoring': False,
  'slm_used_for_s

In [56]:
def render_provider_summary(patient_id: str, cards: List[Dict[str, Any]]) -> str:
    profile = SYNTHETIC_PATIENT_PROFILES[patient_id]
    lines = []
    lines.append(f"UC4 Structured Micro-Priority Provider Summary")
    lines.append(f"Patient: {profile['first_name']} ({patient_id})")
    lines.append(f"Synthetic demo data: True")
    lines.append(f"Generated: {RUN_TIMESTAMP}")
    lines.append("")
    lines.append("Safety boundary:")
    lines.append("- This summary is not a diagnosis.")
    lines.append("- No medication causality is inferred.")
    lines.append("- No treatment or medication changes are recommended.")
    lines.append("- Free text was not used for scoring.")
    lines.append("- SLM output was not used for scoring.")
    lines.append("- UC1/UC2 emergency workflows remain separate.")
    lines.append("")

    if not cards:
        lines.append("No UC4 routine priorities generated.")
        return "\n".join(lines)

    lines.append("Top structured priorities:")
    for idx, card in enumerate(cards, start=1):
        lines.append("")
        lines.append(f"{idx}. {card['title']}")
        lines.append(f"   Template ID: {card['template_id']}")
        lines.append(f"   Score: {card['score']}")
        lines.append(f"   Fired rules: {', '.join([r['rule_code'] for r in card['fired_rules']])}")
        lines.append(f"   Evidence: {summarize_evidence(card['fired_rules'])}")
        lines.append(f"   Provider summary draft: {card['provider_summary_draft']}")

    return "\n".join(lines)

PROVIDER_SUMMARIES = {}
for pid, cards in STRUCTURED_PRIORITY_CARDS_BY_PATIENT.items():
    summary = render_provider_summary(pid, cards)
    PROVIDER_SUMMARIES[pid] = summary
    with open(OUTPUT_DIR / f"uc4_provider_summary_{pid}.txt", "w", encoding="utf-8") as f:
        f.write(summary)

print(PROVIDER_SUMMARIES["Mike_DEMO_001"])

UC4 Structured Micro-Priority Provider Summary
Patient: Mike (Mike_DEMO_001)
Synthetic demo data: True
Generated: 2026-07-16T19:17:58.232930+00:00

Safety boundary:
- This summary is not a diagnosis.
- No medication causality is inferred.
- No treatment or medication changes are recommended.
- Free text was not used for scoring.
- SLM output was not used for scoring.
- UC1/UC2 emergency workflows remain separate.

Top structured priorities:

1. Discomfort during transfers or positioning
   Template ID: TRANSFER_DISCOMFORT_TRACKING
   Score: 0.7
   Fired rules: R_TRANSFER_DISCOMFORT_RECURRENCE, R_TRANSFER_CONTEXT_CLUSTER, R_LOW_MOVEMENT_INCREASE
   Evidence: observation_counts_7d.PAIN_OR_DISCOMFORT=1 (>= 1); observation_counts_7d.TRANSFER_OR_POSITIONING_CONTEXT=1 (>= 1); context_counts_7d.DURING_TRANSFER=1 (>= 1); wearable_deltas.low_movement_delta_7d=2.3 (>= 2.0)
   Provider summary draft: Structured UC4 priority: transfer/positioning discomfort context. Fired rules: R_TRANSFER_DISCOMF

In [58]:
APP_PAYLOADS_BY_PATIENT = {}

for pid, cards in STRUCTURED_PRIORITY_CARDS_BY_PATIENT.items():
    ctx = RULE_CONTEXTS_BY_PATIENT[pid]
    APP_PAYLOADS_BY_PATIENT[pid] = {
        "patient_id": pid,
        "synthetic": True,
        "created_at": RUN_TIMESTAMP,
        "engine_version": ENGINE_VERSION,
        "schema_version": SCHEMA_VERSION,
        "template_registry_version": TEMPLATE_REGISTRY_VERSION,
        "rule_registry_version": RULE_REGISTRY_VERSION,
        "scoring_version": SCORING_VERSION,
        "route": "UC4_ROUTINE_CHECKLIST",
        "safety": {
            "uc1_emergency_active": ctx["safety_flags"]["uc1_emergency_active"],
            "routine_uc4_priorities_generated": len(cards) > 0,
            "free_text_used_for_scoring": False,
            "slm_used_for_scoring": False,
            "diagnostic": False,
            "treatment_recommendation": False,
            "medication_causality": False,
            "emergency_override": False,
        },
        "priority_cards": cards,
    }

write_json(OUTPUT_DIR / "uc4_app_payloads_by_patient.json", APP_PAYLOADS_BY_PATIENT)

In [60]:
AUDIT_RECORDS = []

for pid, cards in STRUCTURED_PRIORITY_CARDS_BY_PATIENT.items():
    for card in cards:
        AUDIT_RECORDS.append({
            "audit_id": f"audit_{pid}_{card['template_id']}",
            "patient_id": pid,
            "synthetic": True,
            "created_at": RUN_TIMESTAMP,
            "event_type": "UC4_PRIORITY_GENERATED",
            "engine_version": ENGINE_VERSION,
            "schema_version": SCHEMA_VERSION,
            "template_registry_version": TEMPLATE_REGISTRY_VERSION,
            "rule_registry_version": RULE_REGISTRY_VERSION,
            "scoring_version": SCORING_VERSION,
            "template_id": card["template_id"],
            "score": card["score"],
            "fired_rule_codes": [r["rule_code"] for r in card["fired_rules"]],
            "evidence_refs": [
                ev
                for r in card["fired_rules"]
                for ev in r.get("evidence_refs", [])
            ],
            "safety": card["safety"],
        })

write_json(OUTPUT_DIR / "uc4_audit_records.json", AUDIT_RECORDS)
pd.DataFrame(AUDIT_RECORDS).to_csv(OUTPUT_DIR / "uc4_audit_records.csv", index=False)
pd.DataFrame(AUDIT_RECORDS).head()

,audit_id,patient_id,synthetic,created_at,event_type,engine_version,schema_version,template_registry_version,rule_registry_version,scoring_version,template_id,score,fired_rule_codes,evidence_refs,safety
0,audit_Mike_DEMO_001_TRANSFER_DISCOMFORT_TRACKING,Mike_DEMO_001,True,2026-07-16T19:17:58.232930+00:00,UC4_PRIORITY_GENERATED,uc4_structured_micropriority_engine_v0.1.0,uc4_schema_v0.1.0,uc4_template_registry_v0.1.0,uc4_rule_registry_v0.1.0,uc4_scoring_v0.1.0,TRANSFER_DISCOMFORT_TRACKING,0.700,"[R_TRANSFER_DISCOMFORT_RECURRENCE, R_TRANSFER_...",[{'field_path': 'observation_counts_7d.PAIN_OR...,"{'diagnosis': False, 'treatment_recommendation..."
1,audit_Mike_DEMO_001_SKIN_PRESSURE_AFTER_SEATED...,Mike_DEMO_001,True,2026-07-16T19:17:58.232930+00:00,UC4_PRIORITY_GENERATED,uc4_structured_micropriority_engine_v0.1.0,uc4_schema_v0.1.0,uc4_template_registry_v0.1.0,uc4_rule_registry_v0.1.0,uc4_scoring_v0.1.0,SKIN_PRESSURE_AFTER_SEATED_PERIOD,0.650,"[R_LOW_MOVEMENT_INCREASE, R_SKIN_PRESSURE_FOCU...",[{'field_path': 'wearable_deltas.low_movement_...,"{'diagnosis': False, 'treatment_recommendation..."
2,audit_Mike_DEMO_001_CAREGIVER_REPORTED_SEIZURE...,Mike_DEMO_001,True,2026-07-16T19:17:58.232930+00:00,UC4_PRIORITY_GENERATED,uc4_structured_micropriority_engine_v0.1.0,uc4_schema_v0.1.0,uc4_template_registry_v0.1.0,uc4_rule_registry_v0.1.0,uc4_scoring_v0.1.0,CAREGIVER_REPORTED_SEIZURE_LIKE_EVENT_CONTEXT,0.650,"[R_UNUSUAL_RESPONSIVENESS_LOGGED, R_CAREGIVER_...",[{'field_path': 'observation_counts_7d.UNUSUAL...,"{'diagnosis': False, 'treatment_recommendation..."
3,audit_JAMES_DEMO_001_FALL_OR_NEAR_FALL_CONTEXT,JAMES_DEMO_001,True,2026-07-16T19:17:58.232930+00:00,UC4_PRIORITY_GENERATED,uc4_structured_micropriority_engine_v0.1.0,uc4_schema_v0.1.0,uc4_template_registry_v0.1.0,uc4_rule_registry_v0.1.0,uc4_scoring_v0.1.0,FALL_OR_NEAR_FALL_CONTEXT,0.650,"[R_TRANSFER_CONTEXT_CLUSTER, R_FALL_OR_NEAR_FA...",[{'field_path': 'context_counts_7d.DURING_TRAN...,"{'diagnosis': False, 'treatment_recommendation..."
4,audit_JAMES_DEMO_001_THERAPY_REHAB_ROUTINE_DIF...,JAMES_DEMO_001,True,2026-07-16T19:17:58.232930+00:00,UC4_PRIORITY_GENERATED,uc4_structured_micropriority_engine_v0.1.0,uc4_schema_v0.1.0,uc4_template_registry_v0.1.0,uc4_rule_registry_v0.1.0,uc4_scoring_v0.1.0,THERAPY_REHAB_ROUTINE_DIFFICULTY,0.585,"[R_THERAPY_ROUTINE_DIFFICULTY, R_REHAB_FOCUS]",[{'field_path': 'observation_counts_7d.THERAPY...,"{'diagnosis': False, 'treatment_recommendation..."


In [62]:
def run_uc1_active_emergency_test() -> Dict[str, Any]:
    ctx = build_rule_context("ELENA_DEMO_001", uc1_emergency_active=True)
    candidates = generate_candidates_for_patient(ctx)

    passed = len(candidates) == 0

    return {
        "test_name": "UC1 active emergency pauses routine UC4 priorities",
        "patient_id": "ELENA_DEMO_001",
        "uc1_emergency_active": True,
        "routine_uc4_priorities_generated": len(candidates) > 0,
        "passed": passed,
        "expected": "No routine UC4 priorities during active UC1 emergency",
        "actual_candidate_count": len(candidates),
    }

def run_uc2_severity2_followup_test() -> Dict[str, Any]:
    ctx = build_rule_context("SOFIA_DEMO_001", uc1_emergency_active=False)
    candidates = generate_candidates_for_patient(ctx)

    has_uc2 = len(ctx["uc2_recent_events"]) >= 1
    generated = len(candidates) > 0
    no_severity3 = all(c["route"] == "UC4_ROUTINE_CHECKLIST" for c in candidates)
    no_slm_scoring = all(c["slm_used_for_scoring"] is False for c in candidates)
    no_free_text_scoring = all(c["free_text_used_for_scoring"] is False for c in candidates)

    passed = has_uc2 and generated and no_severity3 and no_slm_scoring and no_free_text_scoring

    return {
        "test_name": "UC2 severity 2 event can feed UC4 follow-up without emergency override",
        "patient_id": "SOFIA_DEMO_001",
        "has_recent_uc2_event": has_uc2,
        "routine_uc4_priorities_generated": generated,
        "no_emergency_override": no_severity3,
        "no_slm_scoring": no_slm_scoring,
        "no_free_text_scoring": no_free_text_scoring,
        "passed": passed,
        "candidate_template_ids": [c["template_id"] for c in candidates],
    }

UC1_UC2_COMPATIBILITY_TESTS = [
    run_uc1_active_emergency_test(),
    run_uc2_severity2_followup_test(),
]

write_json(OUTPUT_DIR / "uc1_uc2_compatibility_tests.json", UC1_UC2_COMPATIBILITY_TESTS)
UC1_UC2_COMPATIBILITY_TESTS

[{'test_name': 'UC1 active emergency pauses routine UC4 priorities',
  'patient_id': 'ELENA_DEMO_001',
  'uc1_emergency_active': True,
  'routine_uc4_priorities_generated': False,
  'passed': True,
  'expected': 'No routine UC4 priorities during active UC1 emergency',
  'actual_candidate_count': 0},
 {'test_name': 'UC2 severity 2 event can feed UC4 follow-up without emergency override',
  'patient_id': 'SOFIA_DEMO_001',
  'has_recent_uc2_event': True,
  'routine_uc4_priorities_generated': True,
  'no_emergency_override': True,
  'no_slm_scoring': True,
  'no_free_text_scoring': True,
  'passed': True,
  'candidate_template_ids': ['BOWEL_ROUTINE_DISCOMFORT_CONTEXT',
   'MISSED_DELAYED_MEDICATION_CONTEXT',
   'MEDICATION_WINDOW_FATIGUE_TRACKING',
   'CAREGIVER_PROVIDER_REVIEW_REQUEST',
   'SKIN_PRESSURE_AFTER_SEATED_PERIOD']}]

In [64]:
def validate_templates(template_registry: Dict[str, Any]) -> List[str]:
    errors = []

    required_template_keys = [
        "template_id",
        "template_version",
        "title",
        "priority_type",
        "caregiver_summary_template",
        "provider_summary_template",
        "what_to_log_next_schema",
        "safety_flags",
    ]

    for tid, template in template_registry.items():
        for key in required_template_keys:
            if key not in template:
                errors.append(f"Template {tid} missing key: {key}")

        if template.get("template_id") != tid:
            errors.append(f"Template key mismatch for {tid}")

        schema = template.get("what_to_log_next_schema", [])
        if not schema:
            errors.append(f"Template {tid} has empty what_to_log_next_schema")

        for field in schema:
            for fkey in ["field_id", "label", "input_type", "required"]:
                if fkey not in field:
                    errors.append(f"Template {tid} schema field missing {fkey}: {field}")

            if field.get("input_type") in ["select", "multiselect"]:
                if not field.get("options"):
                    errors.append(f"Template {tid} select field missing options: {field.get('field_id')}")

        safety = template.get("safety_flags", {})
        for flag in [
            "diagnosis",
            "treatment_recommendation",
            "medication_causality",
            "emergency_override",
            "slm_scoring",
            "free_text_scoring",
        ]:
            if safety.get(flag) is not False:
                errors.append(f"Template {tid} safety flag {flag} must be False")

    return errors

def validate_cards(cards_by_patient: Dict[str, List[Dict[str, Any]]]) -> List[str]:
    errors = []

    for pid, cards in cards_by_patient.items():
        for card in cards:
            tid = card.get("template_id")

            if tid not in TEMPLATE_REGISTRY:
                errors.append(f"Card for {pid} has unknown template_id: {tid}")
                continue

            if not card.get("what_to_log_next_schema"):
                errors.append(f"Card for {pid}/{tid} has empty schema")

            schema_field_ids = {f["field_id"] for f in TEMPLATE_REGISTRY[tid]["what_to_log_next_schema"]}
            card_schema_field_ids = {f["field_id"] for f in card["what_to_log_next_schema"]}

            if schema_field_ids != card_schema_field_ids:
                errors.append(f"Card schema mismatch for {pid}/{tid}")

            safety = card.get("safety", {})
            for flag in [
                "diagnosis",
                "treatment_recommendation",
                "medication_causality",
                "emergency_override",
                "slm_used_for_scoring",
                "free_text_used_for_scoring",
            ]:
                if safety.get(flag) is not False:
                    errors.append(f"Card {pid}/{tid} safety flag {flag} must be False")

            unsafe_hits = contains_unsafe_language(card.get("caregiver_summary", ""))
            if unsafe_hits:
                errors.append(f"Unsafe caregiver language in {pid}/{tid}: {unsafe_hits}")

            for rule in card.get("fired_rules", []):
                rsafety = rule.get("safety_tags", {})
                for flag in [
                    "diagnostic",
                    "treatment_recommendation",
                    "medication_causality",
                    "emergency_override",
                    "slm_scoring",
                    "free_text_scoring",
                ]:
                    if rsafety.get(flag) is not False:
                        errors.append(f"Fired rule {rule.get('rule_code')} has unsafe flag {flag}")

    return errors

def validate_caregiver_responses(response_examples: List[Dict[str, Any]]) -> List[str]:
    errors = []

    for resp in response_examples:
        tid = resp["source_priority_template_id"]
        if tid not in TEMPLATE_REGISTRY:
            errors.append(f"Response references unknown template: {tid}")
            continue

        schema_field_ids = {f["field_id"] for f in TEMPLATE_REGISTRY[tid]["what_to_log_next_schema"]}
        response_field_ids = set(resp["responses"].keys())

        if not response_field_ids.issubset(schema_field_ids):
            errors.append(f"Response for {tid} has fields not in schema: {response_field_ids - schema_field_ids}")

        if resp.get("free_text_used_for_scoring") is not False:
            errors.append(f"Response for {tid} uses free text for scoring")

        if resp.get("slm_used_for_scoring") is not False:
            errors.append(f"Response for {tid} uses SLM for scoring")

    return errors

VALIDATION_REPORT = {
    "created_at": RUN_TIMESTAMP,
    "engine_version": ENGINE_VERSION,
    "schema_version": SCHEMA_VERSION,
    "template_registry_version": TEMPLATE_REGISTRY_VERSION,
    "rule_registry_version": RULE_REGISTRY_VERSION,
    "scoring_version": SCORING_VERSION,
    "template_errors": validate_templates(TEMPLATE_REGISTRY),
    "rule_registry_validation": rule_registry_validation,
    "card_errors": validate_cards(STRUCTURED_PRIORITY_CARDS_BY_PATIENT),
    "caregiver_response_errors": validate_caregiver_responses(CAREGIVER_RESPONSE_EXAMPLES),
    "uc1_uc2_compatibility_tests": UC1_UC2_COMPATIBILITY_TESTS,
}

VALIDATION_REPORT["passed"] = (
    len(VALIDATION_REPORT["template_errors"]) == 0
    and rule_registry_validation["valid"] is True
    and len(VALIDATION_REPORT["card_errors"]) == 0
    and len(VALIDATION_REPORT["caregiver_response_errors"]) == 0
    and all(t["passed"] for t in UC1_UC2_COMPATIBILITY_TESTS)
)

write_json(OUTPUT_DIR / "uc4_validation_report.json", VALIDATION_REPORT)
VALIDATION_REPORT

{'created_at': '2026-07-16T19:17:58.232930+00:00',
 'engine_version': 'uc4_structured_micropriority_engine_v0.1.0',
 'schema_version': 'uc4_schema_v0.1.0',
 'template_registry_version': 'uc4_template_registry_v0.1.0',
 'rule_registry_version': 'uc4_rule_registry_v0.1.0',
 'scoring_version': 'uc4_scoring_v0.1.0',
 'template_errors': [],
 'rule_registry_validation': {'valid': True,
  'errors': [],
  'warnings': [],
  'rule_count': 23,
  'template_count': 11},
 'card_errors': ["Unsafe caregiver language in Mike_DEMO_001/CAREGIVER_REPORTED_SEIZURE_LIKE_EVENT_CONTEXT: ['\\\\bdiagnos']",
  "Unsafe caregiver language in ELENA_DEMO_001/MEDICATION_WINDOW_FATIGUE_TRACKING: ['\\\\bmedication caused\\\\b']"],
 'caregiver_response_errors': [],
 'uc1_uc2_compatibility_tests': [{'test_name': 'UC1 active emergency pauses routine UC4 priorities',
   'patient_id': 'ELENA_DEMO_001',
   'uc1_emergency_active': True,
   'routine_uc4_priorities_generated': False,
   'passed': True,
   'expected': 'No routin

In [66]:
for pid, cards in STRUCTURED_PRIORITY_CARDS_BY_PATIENT.items():
    print("=" * 100)
    print(pid, "-", SYNTHETIC_PATIENT_PROFILES[pid]["first_name"])
    print("=" * 100)
    for idx, card in enumerate(cards, start=1):
        print(f"\n{idx}. {card['title']}")
        print(f"Template: {card['template_id']}")
        print(f"Score: {card['score']}")
        print(f"Summary: {card['caregiver_summary']}")
        print("What to log next:")
        for field in card["what_to_log_next_schema"]:
            print(f"  - {field['field_id']}: {field['label']}")
        print(f"Fired rules: {', '.join([r['rule_code'] for r in card['fired_rules']])}")
    print("\n")

Mike_DEMO_001 - Mike

1. Discomfort during transfers or positioning
Template: TRANSFER_DISCOMFORT_TRACKING
Score: 0.7
Summary: Recent observations suggest it may be useful to track whether Mike shows discomfort during transfers or positioning.
What to log next:
  - transfer_phase: When did discomfort appear?
  - position_before: Position before the transfer or repositioning?
  - position_after: Position after the transfer or repositioning?
  - discomfort_cues: What discomfort cues did you notice?
  - repositioning_helped: Did repositioning seem to help?
Fired rules: R_TRANSFER_DISCOMFORT_RECURRENCE, R_TRANSFER_CONTEXT_CLUSTER, R_LOW_MOVEMENT_INCREASE

2. Skin and pressure check after seated time
Template: SKIN_PRESSURE_AFTER_SEATED_PERIOD
Score: 0.65
Summary: Mike's care profile includes skin and pressure prevention as a watch area. This week, check whether any skin or pressure concerns appear after longer seated or positioned periods.
What to log next:
  - position_before_check: Where

In [68]:
OUTPUT_MANIFEST = {
    "created_at": RUN_TIMESTAMP,
    "notebook": "UC4_Structured_MicroPriority_RuleRegistry_Validation.ipynb",
    "output_dir": str(OUTPUT_DIR),
    "engine_version": ENGINE_VERSION,
    "schema_version": SCHEMA_VERSION,
    "template_registry_version": TEMPLATE_REGISTRY_VERSION,
    "rule_registry_version": RULE_REGISTRY_VERSION,
    "scoring_version": SCORING_VERSION,
    "validation_passed": VALIDATION_REPORT["passed"],
    "files": sorted([p.name for p in OUTPUT_DIR.iterdir() if p.is_file()]),
}

write_json(OUTPUT_DIR / "output_manifest.json", OUTPUT_MANIFEST)
OUTPUT_MANIFEST

{'created_at': '2026-07-16T19:17:58.232930+00:00',
 'notebook': 'UC4_Structured_MicroPriority_RuleRegistry_Validation.ipynb',
 'output_dir': 'uc4_rule_registry_validation_outputs',
 'engine_version': 'uc4_structured_micropriority_engine_v0.1.0',
 'schema_version': 'uc4_schema_v0.1.0',
 'template_registry_version': 'uc4_template_registry_v0.1.0',
 'rule_registry_version': 'uc4_rule_registry_v0.1.0',
 'scoring_version': 'uc4_scoring_v0.1.0',
 'validation_passed': False,
 'files': ['previous_uc4_priorities.json',
  'synthetic_medication_profiles.csv',
  'synthetic_medication_profiles.json',
  'synthetic_patient_profiles.json',
  'synthetic_shared_care_events.csv',
  'synthetic_wearable_weekly_summaries.json',
  'uc1_uc2_compatibility_tests.json',
  'uc4_app_payloads_by_patient.json',
  'uc4_audit_records.csv',
  'uc4_audit_records.json',
  'uc4_candidates_by_patient.json',
  'uc4_caregiver_response_examples.json',
  'uc4_controlled_vocabularies.json',
  'uc4_micro_priority_candidates_rule